In [2]:
import pandas as pd
import pickle
import torch
from torch.utils.data import DataLoader, Subset
import os
import numpy as np
import sys
# Aggiunge la cartella src al path
sys.path.append(os.path.abspath('../src'))
# Ora puoi importare normalmente
from data_class import Nuc_Dataset
#from utils import training_and_validation_loop_classification, collate_fn,test_classification,training_validation_and_test_loop_classification,find_best_threshold
from utils import test_classification,training_validation_and_test_loop_classification


#from model import CadmusDNA,TransformerNuc_Cadmus


from collections import Counter

from sklearn.model_selection import StratifiedKFold, train_test_split
import math
import seaborn as sns
import matplotlib.pyplot as plt
from collections import defaultdict
from Bio.Seq import Seq
import random



import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import gc


In [3]:

def training_validation_and_test_loop_classification(
    model, dataloader_train, dataloader_validation, dataloader_test,
    epochs=20, lr=0.001, patience=10, weight_decay=0, weigth_dict=None
):
    # Assicurati che il modello sia sul device desiderato
    device = next(model.parameters()).device

    criterion = nn.BCEWithLogitsLoss(weight=weigth_dict)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    train_mcc_list, val_mcc_list, test_mcc_list = [], [], []
    loss_train, loss_val, loss_test = [], [], []

    best_state_cpu = None  # salviamo lo state_dict su CPU per evitare copie GPU
    best_val_loss, best_epoch = float('inf'), 0

    for epoch in range(epochs):
        # === TRAINING ===
        model.train()
        total_loss, batch_count = 0.0, 0
        all_probs, all_labels = [], []

        for batch in dataloader_train:
            optimizer.zero_grad()

            # output_model_from_batch_final deve restituire tensori sul device corretto
            output, output_rc, importance, importance_rc, labels = output_model_from_batch_final(batch, model, device)

            # assicurati che labels siano sul device
            labels = labels.float().to(device)

            # calcolo loss (training)
            loss = criterion(output, labels) + criterion(output_rc, labels)

            if torch.isnan(loss) or loss.item() < 1e-8:
                # pulizia sicura prima di continuare
                del loss, output, output_rc, importance, importance_rc, labels
                gc.collect()
                torch.cuda.empty_cache()
                continue

            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            batch_count += 1

            # Prendi probabilità e spostale su CPU come numpy (detach per rimuovere grafo)
            probs = torch.sigmoid((output + output_rc) / 2).detach().cpu().numpy()
            all_probs.extend(probs.tolist())
            all_labels.extend(labels.detach().cpu().numpy().tolist())

            # --- Pulizia per evitare leak ---
            # elimina riferimenti a tensori GPU pesanti
            del output, output_rc, importance, importance_rc, labels, loss, probs
            # libera memoria Python/GC
            gc.collect()
            # informa l'allocatore CUDA che può liberare memoria inutilizzata
            torch.cuda.empty_cache()
            # opzionale: sincronizza per sicurezza in debug
            # torch.cuda.synchronize()

        # calcoli training
        train_loss = total_loss / batch_count if batch_count > 0 else 0.0
        loss_train.append(train_loss)

        if len(all_probs) > 0:
            train_preds = (np.array(all_probs) > 0.5).astype(int)
            train_mcc = matthews_corrcoef(all_labels, train_preds)
        else:
            train_mcc = 0.0
        train_mcc_list.append(train_mcc)

        # === VALIDATION ===
        model.eval()
        val_total_loss, val_batches = 0.0, 0
        val_probs, val_labels = [], []

        with torch.no_grad():
            for batch in dataloader_validation:
                output, output_rc, importance, importance_rc, labels = output_model_from_batch_final(batch, model, device)
                labels = labels.float().to(device)

                loss = criterion(output, labels) + criterion(output_rc, labels)

                val_total_loss += loss.item()
                val_batches += 1

                probs = torch.sigmoid((output + output_rc) / 2).detach().cpu().numpy()
                val_probs.extend(probs.tolist())
                val_labels.extend(labels.detach().cpu().numpy().tolist())

                # pulizia temporanei della validazione
                del output, output_rc, importance, importance_rc, labels, loss, probs
                gc.collect()
                torch.cuda.empty_cache()

        val_loss = val_total_loss / val_batches if val_batches > 0 else 0.0
        loss_val.append(val_loss)

        if len(val_probs) > 0:
            val_preds = (np.array(val_probs) > 0.5).astype(int)
            val_mcc = matthews_corrcoef(val_labels, val_preds)
        else:
            val_mcc = 0.0
        val_mcc_list.append(val_mcc)

        # === TEST ===
        test_total_loss, test_batches = 0.0, 0
        test_probs, test_labels = [], []

        with torch.no_grad():
            for batch in dataloader_test:
                output, output_rc, importance, importance_rc, labels = output_model_from_batch_final(batch, model, device)
                labels = labels.float().to(device)

                loss = criterion(output, labels) + criterion(output_rc, labels)

                test_total_loss += loss.item()
                test_batches += 1

                probs = torch.sigmoid((output + output_rc) / 2).detach().cpu().numpy()
                test_probs.extend(probs.tolist())
                test_labels.extend(labels.detach().cpu().numpy().tolist())

                # pulizia temporanei del test
                del output, output_rc, importance, importance_rc, labels, loss, probs
                gc.collect()
                torch.cuda.empty_cache()

        test_loss = test_total_loss / test_batches if test_batches > 0 else 0.0
        loss_test.append(test_loss)

        if len(test_probs) > 0:
            test_preds = (np.array(test_probs) > 0.5).astype(int)
            test_mcc = matthews_corrcoef(test_labels, test_preds)
        else:
            test_mcc = 0.0
        test_mcc_list.append(test_mcc)

        # === LOGGING ===
        print(f"\nEpoch {epoch+1}/{epochs}")
        print(f"Train - Loss: {train_loss:.4f}, MCC: {train_mcc:.4f}")
        print(f"Val   - Loss: {val_loss:.4f}, MCC: {val_mcc:.4f}")
        print(f"Test  - Loss: {test_loss:.4f}, MCC: {test_mcc:.4f}")
        print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.2e}")

        # === EARLY STOPPING e salvataggio del best model (solo state_dict su CPU) ===
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            # salva lo state_dict con i tensori trasferiti su CPU => evita di tenere copie GPU in memoria
            best_state_cpu = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            final_test_probs = test_preds.copy() if len(test_probs) > 0 else []
            best_val_probs = val_probs.copy()
            best_true_val = val_labels.copy()

        if epoch - best_epoch >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

        # Piccola pulizia di fine-epoca
        gc.collect()
        torch.cuda.empty_cache()
        # torch.cuda.synchronize()

    # Se vuoi restituire un "modello", restituisci lo state_dict su CPU (più leggero)
    # se vuoi ricaricarlo in seguito:
    # model.load_state_dict(best_state_cpu)
    epoch_best = best_epoch + 1
    return (
        train_mcc_list, val_mcc_list, loss_train, loss_val,
        best_val_loss, best_state_cpu, epoch_best,
        {'label': val_probs}, {'label': val_labels},
        val_labels, val_probs, test_labels, final_test_probs, best_val_probs, best_true_val
    )



In [4]:


class TransformerNuc_Cadmus(nn.Module):
    def __init__(self, input_dim=2560, num_heads=8, dropout_rate=0.0, 
                 f_activation=nn.ReLU()):
        super(TransformerNuc_Cadmus, self).__init__()

        self.transoformer = MyTransformer(embedding_dim=input_dim, num_heads=num_heads, dropout_rate=dropout_rate, activation=f_activation)
        self.act = f_activation
        
        self.final_ffn = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(input_dim, 512),
            self.act,
            nn.Dropout(dropout_rate),
            nn.Linear(512, 1),
        )

        self.final_linear = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(input_dim, 1),
        )

    def forward(self, seq):
        # # seq: (batch_size, seq_len, feature_dim) → no need to transpose
        out, attention_matrix = self.transoformer(seq)
        out = out[:, 0,:]
        out = self.final_linear(out)

        return torch.squeeze(out), attention_matrix


class CadmusDNA(nn.Module):
    
    def __init__(self, att_module, att_parameters, device=None):
        super().__init__()

        # Imposta il dispositivo
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")
    
        # # Carica modello e tokenizer una sola volta
        # self.tokenizer = AutoTokenizer.from_pretrained(
        #     "InstaDeepAI/nucleotide-transformer-2.5B-multi-species"
        # )
        # self.model_LLM = AutoModel.from_pretrained(
        #     "InstaDeepAI/nucleotide-transformer-2.5B-multi-species",
        #     torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
        # ).to(self.device)
        # self.model_LLM.eval()
    
        # Inizializza i moduli
        self.attention_model = att_module(**att_parameters).to(self.device)
        self.linear_output = nn.Linear(2, 1).to(self.device)
        

    # def batch_embeddings(self, sequences, batch_size=32):
    #     """Calcola gli embedding completi per una lista di sequenze in batch."""
    #     all_embeddings = []
    #     for i in tqdm(range(0, len(sequences), batch_size), disable=True):
    #         batch = sequences[i:i+batch_size]
    #         tokens = self.tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to(self.device)
    #         with torch.no_grad():
    #             outputs = self.model_LLM(**tokens)
    #             # Mantiene tutti gli hidden states (nessun pooling)
    #             batch_emb = outputs.last_hidden_state.cpu()
    #         all_embeddings.extend(batch_emb)
    #     return all_embeddings
        

    def forward(self, seqs):
        #print(seqs)
        #embedding_tot = torch.stack(self.batch_embeddings(seqs)).to(self.device)
        output_att, importance = self.attention_model(seqs)
        return output_att, importance



class MyTransformer(nn.Module):
    def __init__(self, embedding_dim, num_heads, dropout_rate, activation=nn.ReLU()):
        super(MyTransformer, self).__init__()
        self.embedding_dim = embedding_dim
        self.act = activation
        self.positional_encoding = SinusoidalPositionalEncoding(self.embedding_dim, max_len=73)

        self.multihead_attention = nn.MultiheadAttention(
            embed_dim=embedding_dim, 
            num_heads=num_heads, 
            dropout=dropout_rate, 
            batch_first=True
        )

        self.norm1 = nn.LayerNorm(self.embedding_dim)
        self.norm2 = nn.LayerNorm(self.embedding_dim)

        self.pw_ffnn = nn.Sequential(
            nn.Linear(self.embedding_dim, 512),
            self.act,
            nn.Dropout(dropout_rate),
            nn.Linear(512, self.embedding_dim)
        )

    def forward(self, seq):
        # seq: (batch_size, seq_len, embedding_dim)
        seq = self.positional_encoding(seq)
        attn_output, attention_matrix = self.multihead_attention(seq, seq, seq)
        attn_output = self.norm1(attn_output + seq)
        ffn_out = self.pw_ffnn(attn_output)
        out = self.norm2(attn_output + ffn_out)
        return out, attention_matrix


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, embedding_dim, max_len=28):    

        super(SinusoidalPositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, embedding_dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embedding_dim, 2).float() * (-torch.log(torch.tensor(10000.0)) / embedding_dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  
        self.register_buffer('pe', pe) 

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


In [5]:

def output_model_from_batch_final(batch, model, device,rc=True):

    '''Dato un modello pytorch e batch restituisce: output_modello, True labels'''

    embedding_tot = batch['embedding'].float().to(device)
    #embedding_tot = batch['sequence']

    if rc:
        embedding_tot_rc = batch['embedding_rev'].float().to(device) 
        #embedding_tot_rc = batch['sequence_rev']
    
    else:
        embedding_tot_rc = batch['embedding'].float().to(device) 
        #embedding_tot_rc = batch['sequence']
        

    labels = batch['label'].to(device)

    output, importance = model(embedding_tot)
    output_rc, importance_rc = model(embedding_tot_rc)

    return output, output_rc, importance, importance_rc, labels


In [6]:
import torch
from torch.utils.data import Dataset

class Nuc_Dataset(Dataset):
    def __init__(self, data, dim_embedding, drop_last=False):
        self.data = data
        self.dim_embedding = dim_embedding
        self.drop_last = drop_last

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]

        if self.drop_last:
            try:
            
                return {
                'sequence':sample['sequence'],
                'length': torch.tensor(144, dtype=torch.int64),
                # 'embedding_CLS': sample['embedding'][0].clone().detach(),
                'embedding': sample['embedding'][0:-3].clone().detach(),
                # 'embedding_CLS_rev': sample['embedding_rev'][0].clone().detach(),
                'embedding_rev': sample['embedding_rev'][0:-3].clone().detach(),
                'label': torch.tensor(sample['label'], dtype=torch.int64) if not isinstance(sample['label'], torch.Tensor) else sample['label'].clone().detach().to(torch.int64),}
                
            except:
                return {
                'sequence':sample['sequence'],

                'length': torch.tensor(144, dtype=torch.int64),
                # 'embedding_CLS': sample['embedding'][0].clone().detach(),
                'embedding': sample['embedding'][0:-3].clone().detach(),
                # 'embedding_CLS_rev': sample['embedding'][0].clone().detach(),
                'embedding_rev': sample['embedding'][0:-3].clone().detach(),
                'label': torch.tensor(sample['label'], dtype=torch.int64) if not isinstance(sample['label'], torch.Tensor) else sample['label'].clone().detach().to(torch.int64),}
        else:
                
            try:
            
                return {
                'sequence':sample['sequence'],

                'length': torch.tensor(144, dtype=torch.int64),
                # 'embedding_CLS': sample['embedding'][0].clone().detach(),
                'embedding': sample['embedding'].clone().detach(),
                # 'embedding_CLS_rev': sample['embedding_rev'][0].clone().detach(),
                'embedding_rev': sample['embedding_rev'].clone().detach(),
                'label': torch.tensor(sample['label'], dtype=torch.int64) if not isinstance(sample['label'], torch.Tensor) else sample['label'].clone().detach().to(torch.int64),}
                
            except:
                return {
                'sequence':sample['sequence'],

                'length': torch.tensor(144, dtype=torch.int64),
                # 'embedding_CLS': sample['embedding'][0].clone().detach(),
                'embedding': sample['embedding'].clone().detach(),
                # 'embedding_CLS_rev': sample['embedding'][0].clone().detach(),
                'embedding_rev': sample['embedding'].clone().detach(),
                'label': torch.tensor(sample['label'], dtype=torch.int64) if not isinstance(sample['label'], torch.Tensor) else sample['label'].clone().detach().to(torch.int64),}



In [9]:
# -----------------------------------------------------------------
# PARTE 1: IMPORTAZIONI E DEFINIZIONE 'OBJECTIVE' PER OPTUNA
# -----------------------------------------------------------------
import optuna
import torch
import numpy as np
from sklearn.metrics import matthews_corrcoef
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import DataLoader, Subset
import copy # Importato per 'local_data'
from optuna.samplers import TPESampler

# --- INIZIO DEFINIZIONE OBJECTIVE PER OPTUNA ---
#
# ASSICURATI CHE LE SEGUENTI VARIABILI/CLASSI SIANO DEFINITE PRIMA DI QUESTA CELLA:
#
# Variabili:
# - data (la lista di dizionari non filtrata)
# - label_counts (il dizionario con i conteggi delle classi)
#
# Classi e Funzioni:
# - Nuc_Dataset
# - CadmusDNA
# - TransformerNuc_Cadmus
# - training_validation_and_test_loop_classification
# - test_classification (usato nella Parte 3)
#
# -----------------------------------------------------------------


g = torch.Generator()
g.manual_seed(42) # Usa lo stesso seed

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def objective(trial):
    """
    Funzione obiettivo di Optuna.
    Esegue un'intera 5-fold CV e restituisce la media dell'MCC 
    calcolata sui set di *validazione interna* di ogni fold.
    """
    
    # --- 1. Definizione Iperparametri da testare ---
    hp_dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
    hp_num_heads = trial.suggest_categorical('num_heads', [4, 5, 8]) 
    hp_batch_size = trial.suggest_categorical('batch_size', [32, 64])
    hp_lr = trial.suggest_float('lr', 1e-6, 1e-4, log=True)
    hp_weight_decay = trial.suggest_float('weight_decay', 1e-8, 1e-5, log=True)
    hp_patience = trial.suggest_int('patience', 5, 15)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    dim_embedding = 2560
    
    # Crea una copia locale per questo trial
    # (Usiamo deepcopy per sicurezza, anche se il filtro potrebbe essere sufficiente)
    local_data = [x for x in data if x['embedding'].shape[0] == 28]

    if not local_data:
        print("Attenzione: nessun dato rimasto dopo il filtraggio.")
        return -1.0 

    dataset = Nuc_Dataset(local_data, dim_embedding, drop_last=False)
    labels = np.array([dataset[i]['label'].item() for i in range(len(dataset))])
    
    if len(labels) == 0:
        print("Attenzione: nessun label nel dataset.")
        return -1.0 

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    internal_val_mccs = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
        
        train_labels = labels[train_idx]
        
        internal_train_idx, internal_val_idx = train_test_split(
            train_idx,
            test_size=0.05,
            stratify=train_labels,
            random_state=fold
        )

        internal_train_subset = Subset(dataset, internal_train_idx)
        internal_val_subset = Subset(dataset, internal_val_idx)
        
        # --- Usa iperparametri suggeriti da Optuna ---
        dataloader_internal_train = DataLoader(internal_train_subset, batch_size=hp_batch_size, shuffle=True,worker_init_fn=seed_worker,  
    generator=g )
        dataloader_internal_val = DataLoader(internal_val_subset, batch_size=hp_batch_size, shuffle=True,worker_init_fn=seed_worker,
    generator=g)
        
        # Definiamo il test_loader del fold CV (necessario per la funzione di training)
        test_subset_cv = Subset(dataset, val_idx) 
        dataloader_test_cv = DataLoader(test_subset_cv, batch_size=hp_batch_size, shuffle=True,worker_init_fn=seed_worker,
    generator=g)
        
        # --- Modello con iperparametri suggeriti ---
        transf_parameters_att = {'input_dim': dim_embedding, 
                                 'dropout_rate': hp_dropout_rate, 
                                 'num_heads': hp_num_heads}
        
        # Calcolo Pesi
        class_weights = {label: len(local_data) / count for label, count in label_counts.items()}
        max_weight = max(class_weights.values())
        class_weights = {label: weight / max_weight for label, weight in class_weights.items()}
        weights_tensor = torch.tensor([class_weights[0], class_weights[1]], dtype=torch.float32).to(device)
        labels_list = ['label']
        weight_dict = {label: weights_tensor for label in labels_list}
        
        model_internal = CadmusDNA(TransformerNuc_Cadmus, transf_parameters_att, device)
        
        # --- Allenamento ---
        # L'obiettivo è massimizzare la performance su (dataloader_internal_val)
        # I risultati su (dataloader_test_cv) sono ignorati in questa fase
        _, _, _, _, _, _, _, _, _, val_labels, val_probs, _, _ = training_validation_and_test_loop_classification(
            model_internal,
            dataloader_internal_train,
            dataloader_internal_val, # Set di validazione interna
            dataloader_test_cv,      # Set di test del fold (usato come "dummy" qui)
            epochs= 200,             
            lr=hp_lr,
            weight_decay=hp_weight_decay,
            patience=hp_patience
        )

        # --- Calcolo Metrica (MCC) sul set di validazione INTERNA ---
        if len(val_labels) > 0 and len(val_probs) > 0:
            val_preds = (np.array(val_probs) > 0.5).astype(int)
            fold_val_mcc = matthews_corrcoef(val_labels, val_preds)
            internal_val_mccs.append(fold_val_mcc)
        else:
            # Penalizza se il training fallisce o non produce output
            internal_val_mccs.append(-1.0) 

        # --- Pruning (Opzionale ma consigliato) ---
        trial.report(np.mean(internal_val_mccs), fold)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    mean_internal_val_mcc = np.mean(internal_val_mccs)
    return mean_internal_val_mcc

# --- FINE DEFINIZIONE OBJECTIVE ---


# -----------------------------------------------------------------
# PARTE 2: ESECUZIONE DELLO STUDIO OPTUNA
# -----------------------------------------------------------------

print("--- AVVIO STUDIO OPTUNA ---")
# Devi creare un sampler "seminato"
seeded_sampler = TPESampler(seed=42) # Usa lo stesso seed


study = optuna.create_study(
    study_name='my_cv_optimization_cadmus_data1_RIPRODUCIBILE_03_02_26',  # Un nome per il tuo studio
    storage='sqlite:///my_study_cadmus_data1_RIPRODUCIBILE_03_02_26.db',  # Nome del file dove salvare
    load_if_exists=True,              # LA MAGIA: carica i risultati se il file esiste
    direction='maximize',
    sampler=seeded_sampler,  # <-- DI' A OPTUNA DI USARLO
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2)
)



# n_trials=50 significa 50 * 5 = 250 training (costoso!)
# Imposta n_trials a un valore ragionevole (es. 20-50)
study.optimize(objective, n_trials=100) 

print("\n--- STUDIO OPTUNA COMPLETATO ---")
print(f"Miglior trial (Mean Internal Val MCC): {study.best_value}")
print("Iperparametri migliori:")
print(study.best_params)

# Salva i migliori parametri per la Parte 3
best_hyperparameters = study.best_params


# -----------------------------------------------------------------
# PARTE 3: ESECUZIONE FINALE CON I MIGLIORI IPERPARAMETRI
# (Questo è il tuo codice originale, modificato per usare i best_params)
# -----------------------------------------------------------------

print("\n--- AVVIO CROSS-VALIDATION FINALE CON HP OTTIMIZZATI ---")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

dim_embedding = 2560
# Rieseguiamo il filtraggio sui dati originali
data_filtered = [x for x in data if x['embedding'].shape[0] == 28] 

dataset = Nuc_Dataset(data_filtered, dim_embedding, drop_last=False)

labels = np.array([dataset[i]['label'].item() for i in range(len(dataset))])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []
for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
    print(f"\n--- Fold {fold+1} ---")

    train_labels = labels[train_idx]
    
    internal_train_idx, internal_val_idx = train_test_split(
        train_idx,
        test_size=0.05,
        stratify=train_labels,
        random_state=fold
    )

    internal_train_subset = Subset(dataset, internal_train_idx)
    internal_val_subset = Subset(dataset, internal_val_idx)
    
    # --- USA HP OTTIMIZZATI ---
    dataloader_internal_train = DataLoader(internal_train_subset, batch_size=best_hyperparameters['batch_size'], shuffle=True)
    dataloader_internal_val = DataLoader(internal_val_subset, batch_size=best_hyperparameters['batch_size'], shuffle=True)

    test_subset_cv = Subset(dataset, val_idx) # Corretto
    dataloader_test_cv = DataLoader(test_subset_cv, batch_size=best_hyperparameters['batch_size'], shuffle=True)
    
    # --- USA HP OTTIMIZZATI ---
    #transf_parameters_cls = {'input_dim': dim_embedding, 'dropout_rate': 0.}
    transf_parameters_att = {'input_dim': dim_embedding, 
                             'dropout_rate': best_hyperparameters['dropout_rate'], 
                             'num_heads': best_hyperparameters['num_heads']}
    #transf_parameters_ohe = {'input_dim': dim_embedding}
    
    # Calcolo pesi (codice originale)
    class_weights = {label: len(data_filtered) / count for label, count in label_counts.items()}
    max_weight = max(class_weights.values())
    class_weights = {label: weight / max_weight for label, weight in class_weights.items()}
    weights_tensor = torch.tensor([class_weights[0], class_weights[1]], dtype=torch.float32).to(device)
    labels_list = ['label']
    weight_dict = {label: weights_tensor for label in labels_list}
    print(weight_dict)
    
    model_internal = CadmusDNA(TransformerNuc_Cadmus, transf_parameters_att, device)
    
    # --- USA HP OTTIMIZZATI ---
    # Allenamento finale: ora ci interessano le performance su dataloader_test_cv
    _, _, _, _, best_val_acc, best_model, best_epoch, _, _, val_labels, val_probs, test_labels, test_probs = training_validation_and_test_loop_classification(
        model_internal,
        dataloader_internal_train,
        dataloader_internal_val,
        dataloader_test_cv, # Questo è il VERO test set del fold CV
        epochs= 200,
        lr=best_hyperparameters['lr'],
        weight_decay=best_hyperparameters['weight_decay'],
        patience=best_hyperparameters['patience']
    )

    # Salva i risultati del fold (valutati sul test set del fold CV)
    fold_results.append({
        'fold': fold + 1,
        'test_metrics': test_classification(best_model, dataloader_test_cv, threshold=0.5)[0],
        'best_epoch': best_epoch
    })

# Stampa i risultati finali medi (sul test set dei 5 fold)
print("\n--- RISULTATI FINALI CV (su test folds) CON HP OTTIMIZZATI ---")
for metrics in ['Sensitivity (Recall)',
  'Specificity',
  'Accuracy',
  'MCC',
  'AUC',
  'PR AUC',]:
    # Gestisce il caso in cui la metrica non sia presente
    metric_values = [i['test_metrics'].get(metrics) for i in fold_results if i['test_metrics'].get(metrics) is not None]
    if metric_values:
        print(f"{metrics}: {np.mean(metric_values)}")
    else:
        print(f"{metrics}: N/A (Metrica non trovata)")

--- AVVIO STUDIO OPTUNA ---


[I 2026-02-03 15:50:50,936] A new study created in RDB with name: my_cv_optimization_cadmus_data1_RIPRODUCIBILE_03_02_26
[W 2026-02-03 15:50:51,065] Trial 0 failed with parameters: {'dropout_rate': 0.249816047538945, 'num_heads': 4, 'batch_size': 32, 'lr': 1.306673923805328e-06, 'weight_decay': 3.967605077052987e-06, 'patience': 11} because of the following error: NameError("name 'data' is not defined").
Traceback (most recent call last):
  File "/home/guido/miniconda3/envs/nucleosome_env/lib/python3.10/site-packages/optuna/study/_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_1326054/4092557714.py", line 60, in objective
    local_data = [x for x in data if x['embedding'].shape[0] == 28]
NameError: name 'data' is not defined
[W 2026-02-03 15:50:51,066] Trial 0 failed with value None.


NameError: name 'data' is not defined

In [ ]:
assert False

In [7]:
#CARICAMENTO DATI

In [8]:
# #First Dataset

# with open('../data/dataframe_nup_1/dataset_nup1_sapiens.pkl', 'rb') as f:  
#     dataset_sapiens = pickle.load(f)
# with open('../data/dataframe_nup_1/dataset_nup1_sapiens_reverse.pkl', 'rb') as f:
#     dataset_sapiens_rev = pickle.load(f)

# RC_augmentation=True

# if RC_augmentation:
#     for d1, d2 in zip(dataset_sapiens, dataset_sapiens_rev):
#         d1['embedding_rev'] = d2['embedding']  
#         #d1['sequence_rev'] = d2['sequence']  
        
# data = dataset_sapiens

# #conto le occorrenze per ogni classe
# labels = [entry['label'] for entry in data]
# label_counts = Counter(labels)
# for label, count in label_counts.items():
#     print(f"Label {label}: {count} samples")

In [ ]:
#First Dataset

with open('../data/data_pkl/', 'rb') as f:  
    dataset_sapiens = pickle.load(f)
with open('../data/data_pkl/dataset_nup1_melanogaster_reverse.pkl', 'rb') as f:
    dataset_sapiens_rev = pickle.load(f)

RC_augmentation=True

if RC_augmentation:
    for d1, d2 in zip(dataset_sapiens, dataset_sapiens_rev):
        d1['embedding_rev'] = d2['embedding']  
        #d1['sequence_rev'] = d2['sequence']  
        
data = dataset_sapiens

#conto le occorrenze per ogni classe
labels = [entry['label'] for entry in data]
label_counts = Counter(labels)
for label, count in label_counts.items():
    print(f"Label {label}: {count} samples")

In [ ]:
# #Second Dataset PR

# dataset_HS_PR = []

# with open('../data/dataframe_nup_2/dataset_nup2_sapiens_promoter_0_10k.pkl', 'rb') as f:
#     dataset_HS_PR += pickle.load(f)
# with open('../data/dataframe_nup_2/dataset_nup2_sapiens_promoter_10k_20k.pkl', 'rb') as f:
#     dataset_HS_PR += pickle.load(f)
# with open('../data/dataframe_nup_2/dataset_nup2_sapiens_promoter_20k_30k.pkl', 'rb') as f:
#     dataset_HS_PR += pickle.load(f)
# with open('../data/dataframe_nup_2/dataset_nup2_sapiens_promoter_30k_40k.pkl', 'rb') as f:
#     dataset_HS_PR += pickle.load(f)
# with open('../data/dataframe_nup_2/dataset_nup2_sapiens_promoter_40k_50k.pkl', 'rb') as f:
#     dataset_HS_PR += pickle.load(f)
# with open('../data/dataframe_nup_2/dataset_nup2_sapiens_promoter_50k_60k.pkl', 'rb') as f:
#     dataset_HS_PR += pickle.load(f)
# with open('../data/dataframe_nup_2/dataset_nup2_sapiens_promoter_60k_70k.pkl', 'rb') as f:
#     dataset_HS_PR += pickle.load(f)
# with open('../data/dataframe_nup_2/dataset_nup2_sapiens_promoter_70k_80k.pkl', 'rb') as f:
#     dataset_HS_PR += pickle.load(f)
# with open('../data/dataframe_nup_2/dataset_nup2_sapiens_promoter_80k_90k.pkl', 'rb') as f:
#     dataset_HS_PR += pickle.load(f)
# with open('../data/dataframe_nup_2/dataset_nup2_sapiens_promoter_90k_100k.pkl', 'rb') as f:
#     dataset_HS_PR += pickle.load(f)

# # # #quando avrò il RC
# # if RC_augmentation:
# #     for d1, d2 in zip(dataset_HS_PR, dataset_HS_PR_rev):
# #         d1['embedding_rev'] = d2['embedding']


# data = dataset_HS_PR 

# #conto le occorrenze per ogni classe
# labels = [entry['label'] for entry in data]
# label_counts = Counter(labels)
# for label, count in label_counts.items():
#     print(f"Label {label}: {count} samples")


In [ ]:
# #Second Dataset 5U
    
# with open('../data/dataframe_nup_2/dataset_nup2_sapiens_5u.pkl', 'rb') as f:
#     dataset_sapiens_5u = pickle.load(f)
# with open('../data/dataframe_nup_2/dataset_nup2_sapiens_5u_reverse.pkl', 'rb') as f:
#     dataset_sapiens_5u_rev = pickle.load(f)

# if RC_augmentation:
#     for d1, d2 in zip(dataset_sapiens_5u, dataset_sapiens_5u_rev):
#         d1['embedding_rev'] = d2['embedding'] 


# data = dataset_sapiens_5u

# #conto le occorrenze per ogni classe
# labels = [entry['label'] for entry in data]
# label_counts = Counter(labels)
# for label, count in label_counts.items():
#     print(f"Label {label}: {count} samples")

In [ ]:
#Second Dataset Largest C.

#..... Da fare 

In [ ]:
# #Third dataset

# #dataset_nup1_sapiens   dataset_nup1_elegans
# with open('../data/dataframe_nup_3/dataset_nup3_h38.pkl', 'rb') as f:
#     dataset_sapiens_h38 = pickle.load(f)
# with open('../data/dataframe_nup_3/dataset_nup3_h38_reverse.pkl', 'rb') as f:
#     dataset_sapiens_h38_rev = pickle.load(f)

# if RC_augmentation:
#     for d1, d2 in zip(dataset_sapiens_h38, dataset_sapiens_h38_rev):
#         d1['embedding_rev'] = d2['embedding'] 

# data = dataset_sapiens_h38

# #conto le occorrenze per ogni classe
# labels = [entry['label'] for entry in data]
# label_counts = Counter(labels)
# for label, count in label_counts.items():
#     print(f"Label {label}: {count} samples")

In [ ]:


def test_classification(model, dataloader_test, threshold=0.5):
    device = next(model.parameters()).device
    model.eval()

    val_labels, val_preds = [], []
    importance_list = []
    importance_rc_list = []
    with torch.no_grad():
        for batch in dataloader_test:
            output, output_rc, importance, importance_rc, labels= output_model_from_batch_final(batch, model, device)
            # labels = batch['label'].float().to(device)

            probs = torch.sigmoid((output+output_rc)/2).cpu().numpy()

            val_preds.extend(probs)
            val_labels.extend(labels.cpu().numpy())
            importance_list.append(importance)
            importance_rc_list.append(importance_rc)

    metrics = classification_metrics(val_labels, val_preds, threshold=threshold)
    return metrics, val_labels, val_preds, importance_list, importance_rc_list,val_preds

import os
import numpy as np
import pandas as pd  # Assicurati di importare pandas
import matplotlib.pyplot as plt
from sklearn.metrics import (confusion_matrix, recall_score, accuracy_score, 
                             matthews_corrcoef, roc_auc_score, average_precision_score,
                             roc_curve, precision_recall_curve)

def classification_metrics(y_true, y_pred_probs, threshold=0.5):
    """
    Calcola: Sensitivity, Specificity, Accuracy, MCC, AUC, PR AUC,
    salva ROC e PR curve e salva le predizioni in un CSV.
    """

    # Converti in numpy array e appiattisci se necessario (ravel) per evitare errori di forma
    y_true = np.array(y_true).ravel()
    y_pred_probs = np.array(y_pred_probs).ravel()
    y_pred = (y_pred_probs >= threshold).astype(int)

    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    print('final threshold', threshold)

    # Metriche base
    sensitivity = recall_score(y_true, y_pred)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    accuracy = accuracy_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    auc = roc_auc_score(y_true, y_pred_probs)
    pr_auc = average_precision_score(y_true, y_pred_probs)

    # ---------- Creazione cartella ../data se non esiste ----------
    save_dir = "../data"
    os.makedirs(save_dir, exist_ok=True)

    # ==================== SALVATAGGIO CSV ====================
    # Creiamo un DataFrame con True, Probabilities e Predicted Class
    df_results = pd.DataFrame({
        'y_true': y_true,
        'y_pred_probs': y_pred_probs,  # Molto utile per analisi post-hoc
        'y_pred': y_pred               # La classe finale basata sulla threshold
    })
    
    csv_path = os.path.join(save_dir, "predictions.csv")
    df_results.to_csv(csv_path, index=False)
    print(f"File CSV con predizioni salvato in {csv_path}")

    # ==================== ROC CURVE ====================
    fpr, tpr, _ = roc_curve(y_true, y_pred_probs)

    plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}", linewidth=2)
    plt.plot([0, 1], [0, 1], "k--", alpha=0.4)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate (Sensitivity)")
    plt.title("ROC Curve")
    plt.legend(loc="lower right")
    plt.tight_layout()

    roc_path = os.path.join(save_dir, "roc_AUC.png")
    plt.savefig(roc_path, dpi=300)
    plt.close()
    print(f"ROC curve salvata in {roc_path}")

    # ==================== PR CURVE ====================
    precision, recall, _ = precision_recall_curve(y_true, y_pred_probs)

    plt.figure(figsize=(6, 6))
    plt.plot(recall, precision, linewidth=2)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"Precision-Recall Curve (AP = {pr_auc:.3f})")
    plt.tight_layout()

    pr_path = os.path.join(save_dir, "pr_curve.png")
    plt.savefig(pr_path, dpi=300)
    plt.close()
    print(f"PR curve salvata in {pr_path}")

    # Output metriche
    output = {
        "Sensitivity (Recall)": sensitivity,
        "Specificity": specificity,
        "Accuracy": accuracy,
        "MCC": mcc,
        "AUC": auc,
        "PR AUC": pr_auc
    }

    print(output)
    return output


In [ ]:
#FINE CARICAMENTO DATI

In [ ]:
from sklearn.metrics import matthews_corrcoef


In [ ]:
def set_seed(seed=42):
    random.seed(seed)                          # Seed Python (modulo random)
    np.random.seed(seed)                       # Seed NumPy
    torch.manual_seed(seed)                    # Seed PyTorch CPU
    torch.cuda.manual_seed(seed)               # Seed PyTorch CUDA (una GPU)
    torch.cuda.manual_seed_all(seed)           # Seed PyTorch CUDA (tutte le GPU)

    torch.backends.cudnn.deterministic = True  # Usa algoritmi deterministici
    torch.backends.cudnn.benchmark = False     # Disabilita ottimizzazioni non deterministiche

    print(f"Seeds fixed with seed = {seed}")
set_seed()

In [ ]:
assert False

In [ ]:
# -----------------------------------------------------------------
# PARTE 3: ESECUZIONE FINALE CON I MIGLIORI IPERPARAMETRI
# (Questo è il tuo codice originale, modificato per usare i best_params)
# -----------------------------------------------------------------

best_hyperparameters = {'dropout_rate': 0.3143462158665756,
 'num_heads': 8,
 'batch_size': 32,
 'lr': 1.4271180164633878e-06,
 'weight_decay': 1.2306663346888495e-06,
 'patience': 12}


print("\n--- AVVIO CROSS-VALIDATION FINALE CON HP OTTIMIZZATI ---")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

dim_embedding = 2560
# Rieseguiamo il filtraggio sui dati originali
data_filtered = [x for x in data if x['embedding'].shape[0] == 28] 

dataset = Nuc_Dataset(data_filtered, dim_embedding, drop_last=False)

labels = np.array([dataset[i]['label'].item() for i in range(len(dataset))])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []
for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
    print(f"\n--- Fold {fold+1} ---")

    train_labels = labels[train_idx]
    
    # internal_train_idx, internal_val_idx = train_test_split(
    #     train_idx,
    #     test_size=0.05,
    #     stratify=train_labels,
    #     random_state=fold
    # )

    # internal_train_subset = Subset(dataset, internal_train_idx)
    # internal_val_subset = Subset(dataset, internal_val_idx)
    
    
    # # --- USA HP OTTIMIZZATI ---
    # dataloader_internal_train = DataLoader(internal_train_subset, batch_size=best_hyperparameters['batch_size'], shuffle=True)
    # dataloader_internal_val = DataLoader(internal_val_subset, batch_size=best_hyperparameters['batch_size'], shuffle=True)
    train_subset_cv = Subset(dataset, train_idx) # Corretto
    dataloader_train_cv = DataLoader(train_subset_cv, batch_size=best_hyperparameters['batch_size'], shuffle=True)
    
    test_subset_cv = Subset(dataset, val_idx) # Corretto
    dataloader_test_cv = DataLoader(test_subset_cv, batch_size=best_hyperparameters['batch_size'], shuffle=True)
    
    # --- USA HP OTTIMIZZATI ---
    #transf_parameters_cls = {'input_dim': dim_embedding, 'dropout_rate': 0.}
    transf_parameters_att = {'input_dim': dim_embedding, 
                             'dropout_rate': best_hyperparameters['dropout_rate'], 
                             'num_heads': best_hyperparameters['num_heads']}
    #transf_parameters_ohe = {'input_dim': dim_embedding}

    labels_c = np.array([train_subset_cv[i]['label'].item() for i in range(len(train_subset_cv))])
    label_counts = Counter(labels_c)

    
    # Calcolo pesi (codice originale)
    class_weights = {label: len(train_subset_cv) / count for label, count in label_counts.items()}
    max_weight = max(class_weights.values())
    class_weights = {label: weight / max_weight for label, weight in class_weights.items()}
    weights_tensor = torch.tensor([class_weights[0], class_weights[1]], dtype=torch.float32).to(device)
    labels_list = ['label']
    weight_dict = {label: weights_tensor for label in labels_list}
    print(weight_dict)
    
    model_internal = CadmusDNA(TransformerNuc_Cadmus, transf_parameters_att, device)
    
    # --- USA HP OTTIMIZZATI ---
    # Allenamento finale: ora ci interessano le performance su dataloader_test_cv
    train_mcc_list, val_mcc_list, loss_train, loss_val,best_val_loss, best_state_cpu, epoch_best,val_probs_d, val_labels_d,val_labels, val_probs, test_labels, final_test_probs, best_val_probs, best_true_val = training_validation_and_test_loop_classification(
        model_internal,
        dataloader_train_cv,
        dataloader_test_cv,
        dataloader_test_cv, # Questo è il VERO test set del fold CV
        epochs= 200,
        lr=best_hyperparameters['lr'],
        weight_decay=best_hyperparameters['weight_decay'],
        patience=best_hyperparameters['patience']
    )
    
    # 1. Carica i pesi dentro model_internal (la modifica avviene direttamente sull'oggetto)
    model_internal.load_state_dict(best_state_cpu)
    
    # 2. Ora model_internal contiene i pesi migliori.
    # Se vuoi usare il nome 'best_model' per chiarezza, fai semplicemente un alias:
    best_model = model_internal

    
    # Salva i risultati del fold (valutati sul test set del fold CV)
    fold_results.append({
        'fold': fold + 1,
        'test_metrics': test_classification(best_model, dataloader_test_cv, threshold=0.5)[0],
        'best_epoch': epoch_best
    })

# Stampa i risultati finali medi (sul test set dei 5 fold)
print("\n--- RISULTATI FINALI CV (su test folds) CON HP OTTIMIZZATI ---")
for metrics in ['Sensitivity (Recall)',
  'Specificity',
  'Accuracy',
  'MCC',
  'AUC',
  'PR AUC',]:
    # Gestisce il caso in cui la metrica non sia presente
    metric_values = [i['test_metrics'].get(metrics) for i in fold_results if i['test_metrics'].get(metrics) is not None]
    if metric_values:
        print(f"{metrics}: {np.mean(metric_values)}")
    else:
        print(f"{metrics}: N/A (Metrica non trovata)")

In [ ]:
pd.DataFrame(fold_results).to_pickle('data_1_dm_11_12_25.pkl')

In [ ]:
assert False

In [ ]:
# -----------------------------------------------------------------
# PARTE 3: ESECUZIONE FINALE CON I MIGLIORI IPERPARAMETRI
# (Questo è il tuo codice originale, modificato per usare i best_params)
# -----------------------------------------------------------------

best_hyperparameters = {'dropout_rate': 0.3143462158665756,
 'num_heads': 8,
 'batch_size': 32,
 'lr': 1.4271180164633878e-06,
 'weight_decay': 1.2306663346888495e-06,
 'patience': 12}


print("\n--- AVVIO CROSS-VALIDATION FINALE CON HP OTTIMIZZATI ---")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

dim_embedding = 2560
# Rieseguiamo il filtraggio sui dati originali
data_filtered = [x for x in data if x['embedding'].shape[0] == 28] 

dataset = Nuc_Dataset(data_filtered, dim_embedding, drop_last=False)

labels = np.array([dataset[i]['label'].item() for i in range(len(dataset))])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []
for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
    print(f"\n--- Fold {fold+1} ---")

    train_labels = labels[train_idx]
    
    internal_train_idx, internal_val_idx = train_test_split(
        train_idx,
        test_size=0.05,
        stratify=train_labels,
        random_state=fold
    )

    internal_train_subset = Subset(dataset, internal_train_idx)
    internal_val_subset = Subset(dataset, internal_val_idx)
    
    # --- USA HP OTTIMIZZATI ---
    dataloader_internal_train = DataLoader(internal_train_subset, batch_size=best_hyperparameters['batch_size'], shuffle=True)
    dataloader_internal_val = DataLoader(internal_val_subset, batch_size=best_hyperparameters['batch_size'], shuffle=True)

    test_subset_cv = Subset(dataset, val_idx) # Corretto
    dataloader_test_cv = DataLoader(test_subset_cv, batch_size=best_hyperparameters['batch_size'], shuffle=True)
    
    # --- USA HP OTTIMIZZATI ---
    #transf_parameters_cls = {'input_dim': dim_embedding, 'dropout_rate': 0.}
    transf_parameters_att = {'input_dim': dim_embedding, 
                             'dropout_rate': best_hyperparameters['dropout_rate'], 
                             'num_heads': best_hyperparameters['num_heads']}
    #transf_parameters_ohe = {'input_dim': dim_embedding}
    
    # Calcolo pesi (codice originale)
    class_weights = {label: len(dataset) / count for label, count in label_counts.items()}
    max_weight = max(class_weights.values())
    class_weights = {label: weight / max_weight for label, weight in class_weights.items()}
    weights_tensor = torch.tensor([class_weights[0], class_weights[1]], dtype=torch.float32).to(device)
    labels_list = ['label']
    weight_dict = {label: weights_tensor for label in labels_list}
    print(weight_dict)
    
    model_internal = CadmusDNA(TransformerNuc_Cadmus, transf_parameters_att, device)
    
    # --- USA HP OTTIMIZZATI ---
    # Allenamento finale: ora ci interessano le performance su dataloader_test_cv
    train_mcc_list, val_mcc_list, loss_train, loss_val,best_val_loss, best_state_cpu, epoch_best,val_probs_d, val_labels_d,val_labels, val_probs, test_labels, final_test_probs, best_val_probs, best_true_val = training_validation_and_test_loop_classification(
        model_internal,
        dataloader_internal_train,
        dataloader_internal_val,
        dataloader_test_cv, # Questo è il VERO test set del fold CV
        epochs= 200,
        lr=best_hyperparameters['lr'],
        weight_decay=best_hyperparameters['weight_decay'],
        patience=best_hyperparameters['patience']
    )
    
    # 1. Carica i pesi dentro model_internal (la modifica avviene direttamente sull'oggetto)
    model_internal.load_state_dict(best_state_cpu)
    
    # 2. Ora model_internal contiene i pesi migliori.
    # Se vuoi usare il nome 'best_model' per chiarezza, fai semplicemente un alias:
    best_model = model_internal

    
    # Salva i risultati del fold (valutati sul test set del fold CV)
    fold_results.append({
        'fold': fold + 1,
        'test_metrics': test_classification(best_model, dataloader_test_cv, threshold=0.5)[0],
        'best_epoch': epoch_best
    })

# Stampa i risultati finali medi (sul test set dei 5 fold)
print("\n--- RISULTATI FINALI CV (su test folds) CON HP OTTIMIZZATI ---")
for metrics in ['Sensitivity (Recall)',
  'Specificity',
  'Accuracy',
  'MCC',
  'AUC',
  'PR AUC',]:
    # Gestisce il caso in cui la metrica non sia presente
    metric_values = [i['test_metrics'].get(metrics) for i in fold_results if i['test_metrics'].get(metrics) is not None]
    if metric_values:
        print(f"{metrics}: {np.mean(metric_values)}")
    else:
        print(f"{metrics}: N/A (Metrica non trovata)")

In [ ]:
assert False

In [ ]:
# 1. Carica i pesi dentro model_internal (la modifica avviene direttamente sull'oggetto)
model_internal.load_state_dict(best_state_cpu)

# 2. Ora model_internal contiene i pesi migliori.
# Se vuoi usare il nome 'best_model' per chiarezza, fai semplicemente un alias:
best_model = model_internal
# Salva i risultati del fold (valutati sul test set del fold CV)
fold_results.append({
    'fold': fold + 1,
    'test_metrics': test_classification(best_model, dataloader_test_cv, threshold=0.5)[0],
    'best_epoch': epoch_best
})

# Stampa i risultati finali medi (sul test set dei 5 fold)
print("\n--- RISULTATI FINALI CV (su test folds) CON HP OTTIMIZZATI ---")
for metrics in ['Sensitivity (Recall)',
  'Specificity',
  'Accuracy',
  'MCC',
  'AUC',
  'PR AUC',]:
    # Gestisce il caso in cui la metrica non sia presente
    metric_values = [i['test_metrics'].get(metrics) for i in fold_results if i['test_metrics'].get(metrics) is not None]
    if metric_values:
        print(f"{metrics}: {np.mean(metric_values)}")
    else:
        print(f"{metrics}: N/A (Metrica non trovata)")

In [ ]:
# # #CV WITH EARLY STOPPING
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# dim_embedding = 2560#1280#2560
# data = [x for x in data if x['embedding'].shape[0] == 28]  #QUI CONTROLLA LUNGHEZZA!!!!
# #dataset_HS_PR = [x for x in dataset_HS_PR if x['embedding'].shape[0] == 28]
# # data = [{'sequence': x['sequence'],
# #         'label': x['label'],
# #         'embedding': x['embedding'][:27,:]} for x in data]  #questo da levareeeeeee

# dataset = Nuc_Dataset(data, dim_embedding, drop_last=False)

# labels = np.array([dataset[i]['label'].item() for i in range(len(dataset))])

# skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# fold_results = []
# for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
#     print(f"\n--- Fold {fold+1} ---")

#     # Divisione fold in train/val
#     train_labels = labels[train_idx]
    
#     # Split interno train/test per selezione epoche
#     internal_train_idx, internal_val_idx = train_test_split(
#         train_idx,
#         test_size=0.05,#0.05,
#         stratify=train_labels,
#         random_state=fold  # diverso per ogni fold
#     )

#     internal_train_subset = Subset(dataset, internal_train_idx)
#     internal_val_subset = Subset(dataset, internal_val_idx)
    
#     dataloader_internal_train = DataLoader(internal_train_subset, batch_size=64, shuffle=True)#, collate_fn=collate_fn)
#     dataloader_internal_val = DataLoader(internal_val_subset, batch_size=64, shuffle=True)#, collate_fn=collate_fn)

#     test_subset_cv = Subset(data, val_idx)
#     dataloader_test_cv = DataLoader(test_subset_cv, batch_size=64, shuffle=True)
    
#     # train_subset_cv = Subset(dataset, train_idx)
#     # dataloader_train_cv = DataLoader(train_subset_cv, batch_size=64, shuffle= True)
    
#     # Modello + Pesi
#     transf_parameters_cls = {'input_dim': dim_embedding, 'dropout_rate': 0.}
#     transf_parameters_att = {'input_dim': dim_embedding, 'dropout_rate': 0.3, 'num_heads': 5}
#     transf_parameters_ohe = {'input_dim': dim_embedding}#'input_dim': 4, 'dropout_rate': 0.3, 'kernel_sizes':[i for i in range(3,30, 5)], 'num_heads': 8}
    
#     # cls_module = None
#     # att_module = None #TransformerNuc_onlyTransformer#TransformerNuc_onlyTransformer#TransformerNuc_onlyTransformer#TransformerNuc_onlyTransformer#TransformerNuc
#     # ohe_module = ConvLSTMCombV2#MultiConv1DModel#Conv1DModel

#     #labels_train_csv = label[]
    

#     # Calcola il peso inversamente proporzionale alla frequenza
#     class_weights = {label: len(data) / count for label, count in label_counts.items()}
    
#     # Normalizza (opzionale ma consigliato)
#     max_weight = max(class_weights.values())
#     class_weights = {label: weight / max_weight for label, weight in class_weights.items()}
    
#     # Converte in tensore (ordinando le classi correttamente)
#     weights_tensor = torch.tensor([class_weights[0], class_weights[1]], dtype=torch.float32).to(device)
    
#     # Crea il dizionario finale
#     labels_list = ['label']
#     weight_dict = {label: weights_tensor for label in labels_list}
    
#     print(weight_dict)
    
#     # equal_weights = torch.tensor([1.0, 1.0])
#     # weight_dict = {label: equal_weights.to(device) for label in labels_list}

#     #model_internal =Attention_Nucleosome_Only_CONV(att_module, ohe_module ,device, transf_parameters_cls, transf_parameters_att, transf_parameters_ohe)
#     model_internal = CadmusDNA(TransformerNuc_Cadmus, transf_parameters_att, device)
    
#     # # Allenamento su train interno per selezione epoche
#     # _, _, _, _, best_val_acc, best_model, best_epoch, _, _, val_labels, val_probs,test_labels, test_probs = training_validation_and_test_loop_classification(
#     #     model_internal,
#     #     dataloader_internal_train,
#     #     dataloader_internal_val,
#     #     dataloader_test_cv,
#     #     epochs= 200,
#     #     lr=1e-5,#1e-5,#0.5e-5,#3e-4,#7.873314056904615e-06
#     #     weight_decay=1e-7,#0.5*1e-5,#1.2e-7
#     #     patience=5
#     # )

#     # Allenamento su train interno per selezione epoche
#     _, _, _, _, best_val_acc, best_model, best_epoch, _, _, val_labels, val_probs,test_labels, test_probs = training_validation_and_test_loop_classification(
#         model_internal,
#         dataloader_internal_train,
#         dataloader_internal_val,
#         dataloader_test_cv,
#         epochs= 200,
#         lr=0.5e-5,#1e-5,#0.5e-5,#3e-4,#7.873314056904615e-06
#         weight_decay=1e-7,#0.5*1e-5,#1.2e-7
#         patience=5
#     )

#     # best_thresh, _, _ = find_best_threshold(val_labels, val_probs, test_labels, test_probs)
#     # print(best_thresh)

#     fold_results.append({
#     'fold': fold + 1,
#     'test_metrics': test_classification(best_model, dataloader_test_cv, threshold=0.5)[0],
#     'best_epoch': best_epoch
#     })

# for metrics in ['Sensitivity (Recall)',
#   'Specificity',
#   'Accuracy',
#   'MCC',
#   'AUC',
#   'PR AUC',]:
#     print(metrics, np.mean([i['test_metrics'][metrics] for i in fold_results]))



In [ ]:
# -----------------------------------------------------------------
# PARTE 1: IMPORTAZIONI E DEFINIZIONE 'OBJECTIVE' PER OPTUNA
# -----------------------------------------------------------------
import optuna
import torch
import numpy as np
from sklearn.metrics import matthews_corrcoef
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import DataLoader, Subset
import copy # Importato per 'local_data'
from optuna.samplers import TPESampler

# --- INIZIO DEFINIZIONE OBJECTIVE PER OPTUNA ---
#
# ASSICURATI CHE LE SEGUENTI VARIABILI/CLASSI SIANO DEFINITE PRIMA DI QUESTA CELLA:
#
# Variabili:
# - data (la lista di dizionari non filtrata)
# - label_counts (il dizionario con i conteggi delle classi)
#
# Classi e Funzioni:
# - Nuc_Dataset
# - CadmusDNA
# - TransformerNuc_Cadmus
# - training_validation_and_test_loop_classification
# - test_classification (usato nella Parte 3)
#
# -----------------------------------------------------------------


g = torch.Generator()
g.manual_seed(42) # Usa lo stesso seed

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def objective(trial):
    """
    Funzione obiettivo di Optuna.
    Esegue un'intera 5-fold CV e restituisce la media dell'MCC 
    calcolata sui set di *validazione interna* di ogni fold.
    """
    
    # --- 1. Definizione Iperparametri da testare ---
    hp_dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
    hp_num_heads = trial.suggest_categorical('num_heads', [4, 5, 8]) 
    hp_batch_size = trial.suggest_categorical('batch_size', [32, 64])
    hp_lr = trial.suggest_float('lr', 1e-6, 1e-4, log=True)
    hp_weight_decay = trial.suggest_float('weight_decay', 1e-8, 1e-5, log=True)
    hp_patience = trial.suggest_int('patience', 5, 15)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    dim_embedding = 2560
    
    # Crea una copia locale per questo trial
    # (Usiamo deepcopy per sicurezza, anche se il filtro potrebbe essere sufficiente)
    local_data = [x for x in data if x['embedding'].shape[0] == 28]

    if not local_data:
        print("Attenzione: nessun dato rimasto dopo il filtraggio.")
        return -1.0 

    dataset = Nuc_Dataset(local_data, dim_embedding, drop_last=False)
    labels = np.array([dataset[i]['label'].item() for i in range(len(dataset))])
    
    if len(labels) == 0:
        print("Attenzione: nessun label nel dataset.")
        return -1.0 

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    internal_val_mccs = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
        
        train_labels = labels[train_idx]
        
        internal_train_idx, internal_val_idx = train_test_split(
            train_idx,
            test_size=0.05,
            stratify=train_labels,
            random_state=fold
        )

        internal_train_subset = Subset(dataset, internal_train_idx)
        internal_val_subset = Subset(dataset, internal_val_idx)
        
        # --- Usa iperparametri suggeriti da Optuna ---
        dataloader_internal_train = DataLoader(internal_train_subset, batch_size=hp_batch_size, shuffle=True,worker_init_fn=seed_worker,  
    generator=g )
        dataloader_internal_val = DataLoader(internal_val_subset, batch_size=hp_batch_size, shuffle=True,worker_init_fn=seed_worker,
    generator=g)
        
        # Definiamo il test_loader del fold CV (necessario per la funzione di training)
        test_subset_cv = Subset(dataset, val_idx) 
        dataloader_test_cv = DataLoader(test_subset_cv, batch_size=hp_batch_size, shuffle=True,worker_init_fn=seed_worker,
    generator=g)
        
        # --- Modello con iperparametri suggeriti ---
        transf_parameters_att = {'input_dim': dim_embedding, 
                                 'dropout_rate': hp_dropout_rate, 
                                 'num_heads': hp_num_heads}
        
        # Calcolo Pesi
        class_weights = {label: len(local_data) / count for label, count in label_counts.items()}
        max_weight = max(class_weights.values())
        class_weights = {label: weight / max_weight for label, weight in class_weights.items()}
        weights_tensor = torch.tensor([class_weights[0], class_weights[1]], dtype=torch.float32).to(device)
        labels_list = ['label']
        weight_dict = {label: weights_tensor for label in labels_list}
        
        model_internal = CadmusDNA(TransformerNuc_Cadmus, transf_parameters_att, device)
        
        # --- Allenamento ---
        # L'obiettivo è massimizzare la performance su (dataloader_internal_val)
        # I risultati su (dataloader_test_cv) sono ignorati in questa fase
        _, _, _, _, _, _, _, _, _, val_labels, val_probs, _, _ = training_validation_and_test_loop_classification(
            model_internal,
            dataloader_internal_train,
            dataloader_internal_val, # Set di validazione interna
            dataloader_test_cv,      # Set di test del fold (usato come "dummy" qui)
            epochs= 200,             
            lr=hp_lr,
            weight_decay=hp_weight_decay,
            patience=hp_patience
        )

        # --- Calcolo Metrica (MCC) sul set di validazione INTERNA ---
        if len(val_labels) > 0 and len(val_probs) > 0:
            val_preds = (np.array(val_probs) > 0.5).astype(int)
            fold_val_mcc = matthews_corrcoef(val_labels, val_preds)
            internal_val_mccs.append(fold_val_mcc)
        else:
            # Penalizza se il training fallisce o non produce output
            internal_val_mccs.append(-1.0) 

        # --- Pruning (Opzionale ma consigliato) ---
        trial.report(np.mean(internal_val_mccs), fold)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    mean_internal_val_mcc = np.mean(internal_val_mccs)
    return mean_internal_val_mcc

# --- FINE DEFINIZIONE OBJECTIVE ---


# -----------------------------------------------------------------
# PARTE 2: ESECUZIONE DELLO STUDIO OPTUNA
# -----------------------------------------------------------------

print("--- AVVIO STUDIO OPTUNA ---")
# Devi creare un sampler "seminato"
seeded_sampler = TPESampler(seed=42) # Usa lo stesso seed


study = optuna.create_study(
    study_name='my_cv_optimization_cadmus_data1_RIPRODUCIBILE_03_02_26',  # Un nome per il tuo studio
    storage='sqlite:///my_study_cadmus_data1_RIPRODUCIBILE_03_02_26.db',  # Nome del file dove salvare
    load_if_exists=True,              # LA MAGIA: carica i risultati se il file esiste
    direction='maximize',
    sampler=seeded_sampler,  # <-- DI' A OPTUNA DI USARLO
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2)
)



# n_trials=50 significa 50 * 5 = 250 training (costoso!)
# Imposta n_trials a un valore ragionevole (es. 20-50)
study.optimize(objective, n_trials=100) 

print("\n--- STUDIO OPTUNA COMPLETATO ---")
print(f"Miglior trial (Mean Internal Val MCC): {study.best_value}")
print("Iperparametri migliori:")
print(study.best_params)

# Salva i migliori parametri per la Parte 3
best_hyperparameters = study.best_params


# -----------------------------------------------------------------
# PARTE 3: ESECUZIONE FINALE CON I MIGLIORI IPERPARAMETRI
# (Questo è il tuo codice originale, modificato per usare i best_params)
# -----------------------------------------------------------------

print("\n--- AVVIO CROSS-VALIDATION FINALE CON HP OTTIMIZZATI ---")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

dim_embedding = 2560
# Rieseguiamo il filtraggio sui dati originali
data_filtered = [x for x in data if x['embedding'].shape[0] == 28] 

dataset = Nuc_Dataset(data_filtered, dim_embedding, drop_last=False)

labels = np.array([dataset[i]['label'].item() for i in range(len(dataset))])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []
for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
    print(f"\n--- Fold {fold+1} ---")

    train_labels = labels[train_idx]
    
    internal_train_idx, internal_val_idx = train_test_split(
        train_idx,
        test_size=0.05,
        stratify=train_labels,
        random_state=fold
    )

    internal_train_subset = Subset(dataset, internal_train_idx)
    internal_val_subset = Subset(dataset, internal_val_idx)
    
    # --- USA HP OTTIMIZZATI ---
    dataloader_internal_train = DataLoader(internal_train_subset, batch_size=best_hyperparameters['batch_size'], shuffle=True)
    dataloader_internal_val = DataLoader(internal_val_subset, batch_size=best_hyperparameters['batch_size'], shuffle=True)

    test_subset_cv = Subset(dataset, val_idx) # Corretto
    dataloader_test_cv = DataLoader(test_subset_cv, batch_size=best_hyperparameters['batch_size'], shuffle=True)
    
    # --- USA HP OTTIMIZZATI ---
    #transf_parameters_cls = {'input_dim': dim_embedding, 'dropout_rate': 0.}
    transf_parameters_att = {'input_dim': dim_embedding, 
                             'dropout_rate': best_hyperparameters['dropout_rate'], 
                             'num_heads': best_hyperparameters['num_heads']}
    #transf_parameters_ohe = {'input_dim': dim_embedding}
    
    # Calcolo pesi (codice originale)
    class_weights = {label: len(data_filtered) / count for label, count in label_counts.items()}
    max_weight = max(class_weights.values())
    class_weights = {label: weight / max_weight for label, weight in class_weights.items()}
    weights_tensor = torch.tensor([class_weights[0], class_weights[1]], dtype=torch.float32).to(device)
    labels_list = ['label']
    weight_dict = {label: weights_tensor for label in labels_list}
    print(weight_dict)
    
    model_internal = CadmusDNA(TransformerNuc_Cadmus, transf_parameters_att, device)
    
    # --- USA HP OTTIMIZZATI ---
    # Allenamento finale: ora ci interessano le performance su dataloader_test_cv
    _, _, _, _, best_val_acc, best_model, best_epoch, _, _, val_labels, val_probs, test_labels, test_probs = training_validation_and_test_loop_classification(
        model_internal,
        dataloader_internal_train,
        dataloader_internal_val,
        dataloader_test_cv, # Questo è il VERO test set del fold CV
        epochs= 200,
        lr=best_hyperparameters['lr'],
        weight_decay=best_hyperparameters['weight_decay'],
        patience=best_hyperparameters['patience']
    )

    # Salva i risultati del fold (valutati sul test set del fold CV)
    fold_results.append({
        'fold': fold + 1,
        'test_metrics': test_classification(best_model, dataloader_test_cv, threshold=0.5)[0],
        'best_epoch': best_epoch
    })

# Stampa i risultati finali medi (sul test set dei 5 fold)
print("\n--- RISULTATI FINALI CV (su test folds) CON HP OTTIMIZZATI ---")
for metrics in ['Sensitivity (Recall)',
  'Specificity',
  'Accuracy',
  'MCC',
  'AUC',
  'PR AUC',]:
    # Gestisce il caso in cui la metrica non sia presente
    metric_values = [i['test_metrics'].get(metrics) for i in fold_results if i['test_metrics'].get(metrics) is not None]
    if metric_values:
        print(f"{metrics}: {np.mean(metric_values)}")
    else:
        print(f"{metrics}: N/A (Metrica non trovata)")

In [ ]:
# -----------------------------------------------------------------
# PARTE 3: ESECUZIONE FINALE CON I MIGLIORI IPERPARAMETRI
# (Questo è il tuo codice originale, modificato per usare i best_params)
# -----------------------------------------------------------------

print("\n--- AVVIO CROSS-VALIDATION FINALE CON HP OTTIMIZZATI ---")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

dim_embedding = 2560
# Rieseguiamo il filtraggio sui dati originali
data_filtered = [x for x in data if x['embedding'].shape[0] == 28] 

dataset = Nuc_Dataset(data_filtered, dim_embedding, drop_last=False)

labels = np.array([dataset[i]['label'].item() for i in range(len(dataset))])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []
for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
    print(f"\n--- Fold {fold+1} ---")

    train_labels = labels[train_idx]
    
    internal_train_idx, internal_val_idx = train_test_split(
        train_idx,
        test_size=0.05,
        stratify=train_labels,
        random_state=fold
    )

    internal_train_subset = Subset(dataset, internal_train_idx)
    internal_val_subset = Subset(dataset, internal_val_idx)
    
    # --- USA HP OTTIMIZZATI ---
    dataloader_internal_train = DataLoader(internal_train_subset, batch_size=best_hyperparameters['batch_size'], shuffle=True)
    dataloader_internal_val = DataLoader(internal_val_subset, batch_size=best_hyperparameters['batch_size'], shuffle=True)

    test_subset_cv = Subset(dataset, val_idx) # Corretto
    dataloader_test_cv = DataLoader(test_subset_cv, batch_size=best_hyperparameters['batch_size'], shuffle=True)
    
    # --- USA HP OTTIMIZZATI ---
    #transf_parameters_cls = {'input_dim': dim_embedding, 'dropout_rate': 0.}
    transf_parameters_att = {'input_dim': dim_embedding, 
                             'dropout_rate': best_hyperparameters['dropout_rate'], 
                             'num_heads': best_hyperparameters['num_heads']}
    #transf_parameters_ohe = {'input_dim': dim_embedding}
    
    # Calcolo pesi (codice originale)
    class_weights = {label: len(data_filtered) / count for label, count in label_counts.items()}
    max_weight = max(class_weights.values())
    class_weights = {label: weight / max_weight for label, weight in class_weights.items()}
    weights_tensor = torch.tensor([class_weights[0], class_weights[1]], dtype=torch.float32).to(device)
    labels_list = ['label']
    weight_dict = {label: weights_tensor for label in labels_list}
    print(weight_dict)
    
    model_internal = CadmusDNA(TransformerNuc_Cadmus, transf_parameters_att, device)
    
    # --- USA HP OTTIMIZZATI ---
    # Allenamento finale: ora ci interessano le performance su dataloader_test_cv
    _, _, _, _, best_val_acc, best_model, best_epoch, _, _, val_labels, val_probs, test_labels, test_probs = training_validation_and_test_loop_classification(
        model_internal,
        dataloader_internal_train,
        dataloader_internal_val,
        dataloader_test_cv, # Questo è il VERO test set del fold CV
        epochs= 200,
        lr=best_hyperparameters['lr'],
        weight_decay=best_hyperparameters['weight_decay'],
        patience=best_hyperparameters['patience']
    )

    # Salva i risultati del fold (valutati sul test set del fold CV)
    fold_results.append({
        'fold': fold + 1,
        'test_metrics': test_classification(best_model, dataloader_test_cv, threshold=0.5)[0],
        'best_epoch': best_epoch
    })

# Stampa i risultati finali medi (sul test set dei 5 fold)
print("\n--- RISULTATI FINALI CV (su test folds) CON HP OTTIMIZZATI ---")
for metrics in ['Sensitivity (Recall)',
  'Specificity',
  'Accuracy',
  'MCC',
  'AUC',
  'PR AUC',]:
    # Gestisce il caso in cui la metrica non sia presente
    metric_values = [i['test_metrics'].get(metrics) for i in fold_results if i['test_metrics'].get(metrics) is not None]
    if metric_values:
        print(f"{metrics}: {np.mean(metric_values)}")
    else:
        print(f"{metrics}: N/A (Metrica non trovata)")

In [ ]:
# Dentro il tuo notebook sul server
import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances

study = optuna.load_study(
    study_name='my_cv_optimization_cadmus_data1_RIPRODUCIBILE',
    storage='sqlite:///my_study_cadmus_data1_RIPRODUCIBILE.db'
)

# Questo grafico apparirà direttamente nel notebook
plot_optimization_history(study).show()

import plotly.io as pio
from IPython.display import display, Markdown
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_intermediate_values,
    plot_parallel_coordinate,
    plot_slice
)

# 1. Manteniamo la configurazione che ti funziona (iframe)
pio.renderers.default = "iframe"

# --- GRAFICO 1: ANDAMENTO ---
display(Markdown("## 1. Storia dell'Ottimizzazione"))
display(Markdown("_Il modello sta ancora imparando o si è stabilizzato?_"))
fig_history = plot_optimization_history(study)
fig_history.show()

# --- GRAFICO 2: IMPORTANZA ---
display(Markdown("---")) # Linea divisoria
display(Markdown("## 2. Importanza dei Parametri"))
display(Markdown("_Quali iperparametri spostano davvero l'ago della bilancia?_"))
try:
    fig_imp = plot_param_importances(study)
    fig_imp.show()
except Exception as e:
    print(f"⚠️ Impossibile calcolare l'importanza (forse pochi trial o manca scikit-learn): {e}")

# --- GRAFICO 3: PRUNING ---
display(Markdown("---"))
display(Markdown("## 3. Pruning e Valori Intermedi"))
display(Markdown("_Dove il MedianPruner ha tagliato i trial scarsi (linee interrotte)?_"))
fig_inter = plot_intermediate_values(study)
fig_inter.show()

# --- GRAFICO 4: RELAZIONI ---
display(Markdown("---"))
display(Markdown("## 4. Coordinate Parallele (Overview)"))
display(Markdown("_Visione d'insieme per vedere i flussi dei parametri migliori._"))
fig_parallel = plot_parallel_coordinate(study)
fig_parallel.show()

# --- GRAFICO 5: DETTAGLIO ---
display(Markdown("---"))
display(Markdown("## 5. Dettaglio per Parametro (Slice Plot)"))
display(Markdown("_Dove si concentrano i punti migliori per ogni singolo parametro?_"))
fig_slice = plot_slice(study)
fig_slice.show()

In [ ]:
study.best_value

In [ ]:
#PER VEDERE LE PERFORMANCE FINALI DEL MODELLO


# -----------------------------------------------------------------
# PARTE 3: ESECUZIONE FINALE CON I MIGLIORI IPERPARAMETRI
# (Questo è il tuo codice originale, modificato per usare i best_params)
# -----------------------------------------------------------------

print("\n--- AVVIO CROSS-VALIDATION FINALE CON HP OTTIMIZZATI ---")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

dim_embedding = 2560
# Rieseguiamo il filtraggio sui dati originali
data_filtered = [x for x in data if x['embedding'].shape[0] == 28] 

dataset = Nuc_Dataset(data_filtered, dim_embedding, drop_last=False)

labels = np.array([dataset[i]['label'].item() for i in range(len(dataset))])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []
for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
    print(f"\n--- Fold {fold+1} ---")

    train_labels = labels[train_idx]
    
    internal_train_idx, internal_val_idx = train_test_split(
        train_idx,
        test_size=0.05,
        stratify=train_labels,
        random_state=fold
    )

    internal_train_subset = Subset(dataset, internal_train_idx)
    internal_val_subset = Subset(dataset, internal_val_idx)
    
    # --- USA HP OTTIMIZZATI ---
    dataloader_internal_train = DataLoader(internal_train_subset, batch_size=best_hyperparameters['batch_size'], shuffle=True)
    dataloader_internal_val = DataLoader(internal_val_subset, batch_size=best_hyperparameters['batch_size'], shuffle=True)

    test_subset_cv = Subset(dataset, val_idx) # Corretto
    dataloader_test_cv = DataLoader(test_subset_cv, batch_size=best_hyperparameters['batch_size'], shuffle=True)
    
    # --- USA HP OTTIMIZZATI ---
    #transf_parameters_cls = {'input_dim': dim_embedding, 'dropout_rate': 0.}
    transf_parameters_att = {'input_dim': dim_embedding, 
                             'dropout_rate': best_hyperparameters['dropout_rate'], 
                             'num_heads': best_hyperparameters['num_heads']}
    #transf_parameters_ohe = {'input_dim': dim_embedding}
    
    # Calcolo pesi (codice originale)
    class_weights = {label: len(data_filtered) / count for label, count in label_counts.items()}
    max_weight = max(class_weights.values())
    class_weights = {label: weight / max_weight for label, weight in class_weights.items()}
    weights_tensor = torch.tensor([class_weights[0], class_weights[1]], dtype=torch.float32).to(device)
    labels_list = ['label']
    weight_dict = {label: weights_tensor for label in labels_list}
    print(weight_dict)
    
    model_internal = CadmusDNA(TransformerNuc_Cadmus, transf_parameters_att, device)
    
    # --- USA HP OTTIMIZZATI ---
    # Allenamento finale: ora ci interessano le performance su dataloader_test_cv
    _, _, _, _, best_val_acc, best_model, best_epoch, _, _, val_labels, val_probs, test_labels, test_probs = training_validation_and_test_loop_classification(
        model_internal,
        dataloader_internal_train,
        dataloader_internal_val,
        dataloader_test_cv, # Questo è il VERO test set del fold CV
        epochs= 200,
        lr=best_hyperparameters['lr'],
        weight_decay=best_hyperparameters['weight_decay'],
        patience=best_hyperparameters['patience']
    )

    # Salva i risultati del fold (valutati sul test set del fold CV)
    fold_results.append({
        'fold': fold + 1,
        'test_metrics': test_classification(best_model, dataloader_test_cv, threshold=0.5)[0],
        'best_epoch': best_epoch
    })

# Stampa i risultati finali medi (sul test set dei 5 fold)
print("\n--- RISULTATI FINALI CV (su test folds) CON HP OTTIMIZZATI ---")
for metrics in ['Sensitivity (Recall)',
  'Specificity',
  'Accuracy',
  'MCC',
  'AUC',
  'PR AUC',]:
    # Gestisce il caso in cui la metrica non sia presente
    metric_values = [i['test_metrics'].get(metrics) for i in fold_results if i['test_metrics'].get(metrics) is not None]
    if metric_values:
        print(f"{metrics}: {np.mean(metric_values)}")
    else:
        print(f"{metrics}: N/A (Metrica non trovata)")

In [ ]:
# Stampa i risultati finali medi (sul test set dei 5 fold)
print("\n--- RISULTATI FINALI CV (su test folds) CON HP OTTIMIZZATI ---")
for metrics in ['Sensitivity (Recall)',
  'Specificity',
  'Accuracy',
  'MCC',
  'AUC',
  'PR AUC',]:
    # Gestisce il caso in cui la metrica non sia presente
    metric_values = [i['test_metrics'].get(metrics) for i in fold_results if i['test_metrics'].get(metrics) is not None]
    if metric_values:
        print(f"{metrics}: {np.mean(metric_values)}")
    else:
        print(f"{metrics}: N/A (Metrica non trovata)")

In [ ]:
best_hyperparameters = {'dropout_rate': 0.3143462158665756,
 'num_heads': 8,
 'batch_size': 32,
 'lr': 1.4271180164633878e-06,
 'weight_decay': 1.2306663346888495e-06,
 'patience': 12}

In [ ]:
best_hyperparameters = {'dropout_rate': 0.3143462158665756,
 'num_heads': 8,
 'batch_size': 32,
 'lr': 1.4271180164633878e-06,
 'weight_decay': 1.2306663346888495e-06,
 'patience': 12}


# #CV WITH EARLY STOPPING

from sklearn.metrics import confusion_matrix, accuracy_score, matthews_corrcoef, roc_auc_score
import numpy as np

def classification_metrics(y_true, probs_pred, threshold=0.5):
    """
    Calcola Sensitivity, Specificity, Accuracy, MCC e AUC
    a partire da y_true (etichette vere) e probs_pred (probabilità predette).
    
    Parametri:
        y_true: array-like di etichette vere (0 o 1)
        probs_pred: array-like di probabilità predette (float tra 0 e 1)
        threshold: soglia di classificazione (default 0.5)
    
    Ritorna:
        dizionario con le metriche richieste
    """
    # Conversione in predizioni binarie
    y_pred = (np.array(probs_pred) >= threshold).astype(int)
    y_true = np.array(y_true)
    
    # Matrice di confusione
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    # Metriche
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    accuracy = accuracy_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    auc = roc_auc_score(y_true, probs_pred)
    
    return {
        "Sensitivity_val": sensitivity,
        "Specificity_val": specificity,
        "Accuracy_val": accuracy,
        "MCC_val": mcc,
        "AUC_val": auc
    }

    

g = torch.Generator()
g.manual_seed(0) # Usa lo stesso seed

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

dim_embedding = 2560#1280#2560
# data = [x for x in data if x['embedding'].shape[0] == 37]#[:3000] 

dataset = Nuc_Dataset(data, dim_embedding, drop_last=False)

labels = np.array([dataset[i]['label'].item() for i in range(len(dataset))])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []
for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
    print(f"\n--- Fold {fold+1} ---")

    # Divisione fold in train/val
    train_labels = labels[train_idx]
    
    # # Split interno train/test per selezione epoche
    # internal_train_idx, internal_val_idx = train_test_split(
    #     train_idx,
    #     test_size=0.05,#0.05,
    #     stratify=train_labels,
    #     random_state=fold  # diverso per ogni fold
    # )

    # internal_train_subset = Subset(dataset, internal_train_idx)
    # internal_val_subset = Subset(dataset, internal_val_idx)
    
    # dataloader_internal_train = DataLoader(internal_train_subset, batch_size=32, shuffle=True)#, collate_fn=collate_fn)
    # dataloader_internal_val = DataLoader(internal_val_subset, batch_size=32, shuffle=True)#, collate_fn=collate_fn)

    train_subset_cv = Subset(data, train_idx)
    dataloader_train_cv = DataLoader(train_subset_cv, batch_size=32, shuffle=True,worker_init_fn=seed_worker,  
    generator=g,drop_last=True )
    
    test_subset_cv = Subset(data, val_idx)
    dataloader_test_cv = DataLoader(test_subset_cv, batch_size=32,shuffle=False,worker_init_fn=seed_worker,  
    generator=g,drop_last=True  )
    
    # train_subset_cv = Subset(dataset, train_idx)
    # dataloader_train_cv = DataLoader(train_subset_cv, batch_size=64, shuffle= True)
    
    # Modello + Pesi
    transf_parameters_att = {'input_dim': dim_embedding, 
                             'dropout_rate': best_hyperparameters['dropout_rate'], 
                             'num_heads': best_hyperparameters['num_heads']}   
    # cls_module = None
    # att_module = None #TransformerNuc_onlyTransformer#TransformerNuc_onlyTransformer#TransformerNuc_onlyTransformer#TransformerNuc_onlyTransformer#TransformerNuc
    # ohe_module = ConvLSTMCombV2#MultiConv1DModel#Conv1DModel

    #labels_train_csv = label[]
    

    # Calcola il peso inversamente proporzionale alla frequenza
    class_weights = {label: len(data) / count for label, count in label_counts.items()}
    
    # Normalizza (opzionale ma consigliato)
    max_weight = max(class_weights.values())
    class_weights = {label: weight / max_weight for label, weight in class_weights.items()}
    
    # Converte in tensore (ordinando le classi correttamente)
    weights_tensor = torch.tensor([class_weights[0], class_weights[1]], dtype=torch.float32).to(device)
    
    # Crea il dizionario finale
    labels_list = ['label']
    weight_dict = {label: weights_tensor for label in labels_list}
    
    print(weight_dict)
    
    # equal_weights = torch.tensor([1.0, 1.0])
    # weight_dict = {label: equal_weights.to(device) for label in labels_list}

    #model_internal =Attention_Nucleosome_Only_CONV(att_module, ohe_module ,device, transf_parameters_cls, transf_parameters_att, transf_parameters_ohe)
    model_internal = CadmusDNA(TransformerNuc_Cadmus, transf_parameters_att, device)

    # Allenamento su train interno per selezione epoche
    _, _, _, _, best_val_acc, best_model, best_epoch, _, _, val_labels, val_probs, test_labels, test_probs, best_val_probs, best_true_val = training_validation_and_test_loop_classification(
        model_internal,
        dataloader_train_cv,
        dataloader_test_cv,
        dataloader_test_cv, # Questo è il VERO test set del fold CV
        epochs= 200,
        lr=best_hyperparameters['lr'],
        weight_decay=best_hyperparameters['weight_decay'],
        patience=best_hyperparameters['patience']
    )
    
    # best_thresh, _, _ = find_best_threshold(val_labels, val_probs, test_labels, test_probs)
    # print(best_thresh)

    fold_results.append({
    'fold': fold + 1,
    'test_metrics': test_classification(best_model, dataloader_test_cv, threshold=0.5)[0],
    'best_epoch': best_epoch
    })
    fold_results[-1].update(classification_metrics(best_true_val, best_val_probs))
    break

for metrics in ["Sensitivity_val",
        "Specificity_val",
        "Accuracy_val",
        "MCC_val",
        "AUC_val"]:
    print(metrics, np.mean([i[metrics] for i in fold_results]))


    

for metrics in ['Sensitivity (Recall)',
  'Specificity',
  'Accuracy',
  'MCC',
  'AUC',
  'PR AUC',]:
    print(metrics, np.mean([i['test_metrics'][metrics] for i in fold_results]))



In [ ]:
for i in dataloader_train_cv:
    print(i)
    break

In [ ]:
list(model_internal.named_parameters())


In [ ]:
classification_metrics(best_true_val, best_val_probs)

In [ ]:

def invert_attention(A):
    """
    Inverte le posizioni da 1 in poi nella matrice di attenzione A,
    lasciando la posizione 0 ([CLS]) invariata.
    
    A: matrice di attenzione (L, L)
    """
    L = A.shape[0]
    assert A.shape[0] == A.shape[1], "La matrice deve essere quadrata"
    
    # Costruisci un nuovo ordine di indici: [0, L-1, L-2, ..., 1]
    perm = [0] + list(range(L-1, 0, -1))
    
    # Applica la permutazione a righe e colonne
    A_corrected = A[np.ix_(perm, perm)]
    
    return A_corrected



In [ ]:
import copy
# 1. Filtra dataset con sequenze lunghezza 28
dataset =  copy.deepcopy([i for i in data]) #if i['embedding'].shape[0] == 28]

# 2. Seleziona dati di test
dataset_test_to_study = [dataset[i] for i in val_idx]  # Assicurati che gli indici siano corretti

# 3. Crea dataloader
test = Nuc_Dataset(dataset_test_to_study, dim_embedding)
test = DataLoader(test, batch_size=64, shuffle=False)

# 4. Inference
metrics, val_labels, val_preds, importance, importance_rc = test_classification(model_internal, test)

# 5. Preds binarie
preds_int = [p > 0.5 for p in val_preds]

# 6. Estrai matrici direzionali
all_matrices_dir = [m for batch in importance for m in batch]  # importance = [B, 28, 28]
all_matrices_rc = [m for batch in importance_rc for m in batch]

all_matrices_tot = all_matrices_dir + all_matrices_rc#[x + invert_attention(y) for x,y in zip(all_matrices_dir,all_matrices_rc)]#

preds_int = preds_int + preds_int

dataset_test_to_study *= 2

# 7. Etichette originali
true_label = val_labels + val_labels

# 8. Filtro per veri negativi
all_matrices_tot_filtered = [
    m for m, p, tl in zip(all_matrices_tot, preds_int, true_label)
    #if p == 0 and tl == 0 and m.shape == (28, 28)
]

# 9. Calcolo media e visualizzazione
if len(all_matrices_tot_filtered) > 0:
    stacked = torch.stack(all_matrices_tot_filtered)
    media = stacked[:,0,:].mean(dim=0).unsqueeze(0)
    media_norm = (media - media.min()) / (media.max() - media.min() + 1e-8)

    plt.figure(figsize=(6, 5))
    sns.heatmap(media_norm.cpu().numpy(), cmap="viridis")
    plt.title("Heatmap normalizzata - Veri negativi (p=0, tl=0)")
    plt.show()
else:
    print("⚠️ Nessun vero negativo trovato con shape [28, 28].")


In [ ]:
import torch
from captum.attr import DeepLiftShap
import torch.nn as nn

from captum.attr import GradientShap


class WrappedModel(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model  # salva il modello originale

    def forward(self, x):
        out = self.model(x)
        # se il tuo modello restituisce una tupla (es. (logits, attn))
        if isinstance(out, tuple):
            out = out[0]  # usa solo i logits
        # eventualmente aggiungi .squeeze() se serve
        return out


# Metti il modello in modalità eval
model_internal.eval()
wrapped_model = WrappedModel(model_internal)


data_positivo = [
    row['embedding'] for row, p, tl in zip(dataset_test_to_study, preds_int, val_labels)
    if p == tl and tl == 1
]

data_negativo = [
    row['embedding'] for row, p, tl in zip(dataset_test_to_study, preds_int, val_labels)
    if p == tl and tl == 0
]



grad_shap = GradientShap(wrapped_model)


# Crea un interprete DeepLiftSHAP
dl_shap = DeepLiftShap(wrapped_model)

# Scegli alcune batch di input da analizzare
# (puoi iterare sul dataloader o prenderne una specifica)
batch = next(iter(test))

# Sposta su GPU se necessario
device = next(model_internal.parameters()).device
#inputs = batch['embedding'][0].unsqueeze(0).to(device)
inputs = torch.stack(data_positivo).to(device)

# Definisci le "baselines" — input di riferimento
# esempio: vettore nullo o media delle feature
#baseline = torch.zeros_like(inputs)#torch.zeros_like(inputs)
baseline = torch.stack(data_negativo).squeeze(1).to(device) 

# baseline = torch.stack([
#     torch.zeros_like(inputs),             # tutto 0
#     torch.ones_like(inputs) * 0.5,        # mezzo
#     torch.randn_like(inputs) * 0.1        # un po’ di rumore
# ]).squeeze(1) 


# oppure puoi usare più baselines (DeepLiftSHAP fa la media)
# baseline = torch.stack([torch.zeros_like(inputs), torch.ones_like(inputs)*0.5])

# Calcolo delle attribuzioni DeepLiftSHAP
#attributions = dl_shap.attribute(inputs, baselines=baseline)
attributions = grad_shap.attribute(inputs, baselines=baseline)

#attributions = dl_shap.attribute(inputs, baselines=baseline)

print("Shape delle attribuzioni:", attributions.shape)
print("Esempio di attribuzioni per la prima sequenza:")
print(attributions[0])


In [ ]:
attributions.cpu().sum(dim=2)[0, :].shape

In [ ]:
import torch
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Somma su dimensione embedding → [batch, seq_len]
attr_per_pos = attributions.sum(dim=2)

# Media e deviazione standard sul batch
mean_attr = attr_per_pos.mean(dim=0)
std_attr  = attr_per_pos.std(dim=0)

# Prepara DataFrame
df = pd.DataFrame({
    'posizione': range(len(mean_attr)),
    'importanza_media': mean_attr.cpu().numpy(),
    'std': std_attr.cpu().numpy()
})

# Barplot con barre di errore ± std
plt.figure(figsize=(10,4))
sns.barplot(
    data=df,
    x='posizione',
    y='importanza_media',
    yerr=df['std'],
    color='skyblue',
    errorbar=None
)


plt.xlabel("Posizione nella sequenza")
plt.ylabel("Attribution media ± std")
plt.title("Importanza media per posizione (GradientSHAP)")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(attributions.cpu().sum(dim=2).sum(0))

In [ ]:
sns.heatmap(attributions.cpu().sum(dim=2).sum(0).unsqueeze(0))
#attributions.cpu().sum(dim=2).shape

In [ ]:
sns.heatmap(attributions[4].cpu().sum(dim=1).unsqueeze(0))

In [ ]:
# #TRAIN FOR N EPOCHS
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# dim_embedding = 2560  # oppure 768, deve essere divisibile per num_heads (es: 2, 4, 8, ...)

# dataset = [i for i in dataset_HS_PR if i['embedding'].shape[0] ==28]

# # Crea il dataset
# dataset_train_nuc = Nuc_Dataset(dataset, dim_embedding)

# # Estrai etichette per la stratificazione
# labels = np.array([dataset_train_nuc[i]['label'].item() for i in range(len(dataset_train_nuc))])

# # Indici dell'intero dataset
# train_idx = np.arange(len(dataset_train_nuc))

# # Split interno train/val per selezione epoche
# internal_train_idx, internal_val_idx = train_test_split(
#     train_idx,
#     test_size=0.2,
#     stratify=labels,  # stratify deve corrispondere a train_idx
#     random_state=1
# )

# # Sottoinsiemi dei dati
# internal_train_subset = Subset(dataset_train_nuc, internal_train_idx)
# internal_val_subset = Subset(dataset_train_nuc, internal_val_idx)

# # Dataloader
# dataloader_internal_train = DataLoader(internal_train_subset, batch_size=64, shuffle=True)#, collate_fn=collate_fn)
# dataloader_internal_val = DataLoader(internal_val_subset, batch_size=64, shuffle=False)#, collate_fn=collate_fn)

# # Parametri del modello
# transf_parameters_cls = {'input_dim': dim_embedding, 'dropout_rate': 0.}
# transf_parameters_att = {
#     'input_dim': dim_embedding,
#     'dropout_rate': 0.,
#     'num_heads': 8  # Deve essere divisore di dim_embedding
# }

# # Inizializzazione moduli
# cls_module = CLSModel
# att_module = TransformerNuc_onlyTransformer
# labels_list = ['label']
# equal_weights = torch.tensor([1.0, 1.0])

# weight_dict = {label: equal_weights.to(device) for label in labels_list}

# # Inizializzazione modello
# model_internal = Attention_Nucleosome(cls_module, att_module, device, transf_parameters_cls, transf_parameters_att)

# # Allenamento interno per selezione epoche
# _, val_acc, _, _, best_val_acc, best_model, best_epoch, _, _ = training_and_validation_loop_classification(
#     model_internal,
#     dataloader_internal_train,
#     dataloader_internal_val,
#     weight_dict,
#     epochs=10,
#     lr=1e-6,
#     patience=300,
# )

# # Stampa risultato
# print(f"Best validation accuracy: {best_val_acc:.4f} at epoch {best_epoch}")

In [ ]:
# # Filtro dataset con sequenze di lunghezza 28
# dataset = [i for i in dataset_melanogasters if i['embedding'].shape[0] == 28]

# # Seleziona i dati di test interni
# dataset_test_to_study = [dataset[i] for i in internal_val_idx]

# # Crea il dataloader
# test = Nuc_Dataset(dataset_test_to_study, dim_embedding)
# test = DataLoader(test, batch_size=64, shuffle=False)

# # Inference
# class_metrics, true_label, preds, aa_matrix = test_classification(model_internal, test)
# aa_matrix_rev = aa_matrix  # Se hai reverse matrix diverse, sostituiscila qui

# # Predizioni binarie
# preds_int = [p > 0.5 for p in preds]

# # Appiattisci la lista di batch [B, 28, 28] → singole matrici [28, 28]
# #all_matrices_dir = [m for batch in aa_matrix for m in batch]
# #all_matrices_rev = [m for batch in aa_matrix_rev for m in batch]
# all_matrices_tot_filtered = sum(aa_matrix) #all_matrices_dir + all_matrices_rev

# # Ripeti labels e predizioni per l’allungamento della lista
# true_label_ext = true_label * 2
# preds_int_ext = preds_int * 2

# # # Filtro per VERI NEGATIVI (pred == 0, label == 0)
# # all_matrices_tot_filtered = [
# #     m for m, p, tl in zip(all_matrices_tot, preds_int_ext, true_label_ext)
# #     #if p == 0 and tl == 0 and m.shape == (28, 28)
# # ]

# # Calcolo media e normalizzazione
# if len(all_matrices_tot_filtered) > 0:
#     stacked = torch.stack(all_matrices_tot_filtered)  # [N, 28, 28]
#     media = stacked.mean(dim=0)                       # [28, 28]
#     media_norm = (media - media.min()) / (media.max() - media.min() + 1e-8)

#     # Visualizzazione heatmap
#     plt.figure(figsize=(6, 5))
#     sns.heatmap(media_norm.cpu().numpy(), cmap="viridis")
#     plt.title("Heatmap normalizzata - Veri negativi (p=0, tl=0)")
#     plt.show()
# else:
#     print("⚠️ Nessun vero negativo trovato con shape [28, 28].")


In [ ]:
# #STUDIO ATTENTION MATRIX

# # dataset = [i for i in dataset_sapiens if i['embedding'].shape[0] ==28][0]

# # dataset_test_to_study = [dataset]
# # dataset_test_to_study[0]['embedding'] *= 0

# # test = Nuc_Dataset(dataset_test_to_study, dim_embedding)
# # test = DataLoader(test, batch_size=64, shuffle=False, collate_fn=collate_fn)



# # 1. Filtra dataset con sequenze lunghezza 28
# dataset = [i for i in dataset_sapiens if i['embedding'].shape[0] == 28]

# # 2. Seleziona dati di test
# dataset_test_to_study = [dataset[i] for i in internal_val_idx]  # Assicurati che gli indici siano corretti

# # 3. Crea dataloader
# test = Nuc_Dataset(dataset_test_to_study, dim_embedding)
# test = DataLoader(test, batch_size=64, shuffle=False)

# # 4. Inference
# metrics, val_labels, val_preds, importance = test_classification(model_internal, test)



# # Inference
# metrics, val_labels, val_preds, importance = test_classification(model_internal, test)
# preds_int= [p > 0.5 for p in val_preds]

# # Appiattisco la lista di batch [64, 28, 28] in singoli [28, 28]
# #all_matrices_dir = [m for m in importance]
# all_matrices_dir = [matrix for batchh in importance for matrix in batchh]

# #all_matrices_tot = all_matrices_dir + all_matrices_rev

# # Filtro dei veri negativi (pred = 0, label = 0)
# all_matrices_tot_filtered = [
#     m for m, p, tl in zip(all_matrices_dir, preds_int, val_labels)
#      # if p == 1 and tl == 1
# ]

# data_reduced = [
#     row for row, p, tl in zip(dataset_test_to_study, preds_int, val_labels)
#     # if p == 1 and tl == 1
# ]



# # Calcolo media e normalizzazione
# if len(all_matrices_dir) > 0:
#     stacked = torch.stack(all_matrices_dir)  # shape: [N, 28, 28]
#     media = stacked.mean(dim=0)        # shape: [28, 28]
#     media_norm = (media - media.min()) / (media.max() - media.min() + 1e-8)

#     # heatmap
#     plt.figure(figsize=(6, 5))
#     # sns.heatmap(torch.mean(media_norm.cpu().unsqueeze(0),dim=1), cmap="viridis")
#     sns.heatmap((media_norm.cpu()), cmap="viridis")
#     # sns.heatmap((all_matrices_tot_filtered[10].cpu()), cmap="viridis")

#     plt.title("Heatmap normalizzata - Veri negativi (p=0, tl=0)")
#     plt.show()
# else:
#     print("⚠️ Nessun dato trovato (p == 0 and tl == 0).")

In [ ]:
# #VERSIONE (TOKEN,POSIZIONE)
# data_reduced = [
#     row for row, p, tl in zip(dataset_test_to_study, preds_int, val_labels)
#     if p == tl and tl == 0 #if p == 0 and tl == 0
# ]

# # 8. Filtro per veri negativi
# all_matrices_tot_filtered = [
#     m for m, p, tl in zip(all_matrices_tot, preds_int, val_labels)
#     if p == tl and tl == 0 #if p == 0 and tl == 0 and m.shape == (28, 28)
# ]
# stacked = torch.stack(all_matrices_tot_filtered)

# lista_dizionari = data_reduced
# attention_tensor = stacked

# def tokenize_dna_sequence(seq, k=6, max_tokens=24):
#     return [seq[i:i+k] for i in range(0, k*max_tokens, k)]

# start_idx = 1
# end_idx = 25
# valid_indices = list(range(start_idx, end_idx))  # 1..24

# tokenpos_attention_totals = defaultdict(float)
# tokenpos_counts = defaultdict(int)

# for i, sample in enumerate(lista_dizionari):
#     sequence = sample['sequence']
#     tokens = tokenize_dna_sequence(sequence)  # 24 token
#     attn_received = attention_tensor[i][0,:]   #.sum(dim=0)  # shape (28,)

#     for pos_in_tensor, pos_in_token in enumerate(valid_indices):
#         token = tokens[pos_in_tensor]
#         attn_value = attn_received[pos_in_token].item()
#         key = (token, pos_in_token)
#         tokenpos_attention_totals[key] += attn_value
#         tokenpos_counts[key] += 1

# # Calcola attenzione media per (token, posizione)
# tokenpos_mean_attention = {
#     k: tokenpos_attention_totals[k] / tokenpos_counts[k]
#     for k in tokenpos_attention_totals
# }

# # Combina simmetricamente: (token, pos) ↔ (RC(token), 23 - pos)
# merged_symmetric_scores = {}

# for (token, pos), score in tokenpos_mean_attention.items():
#     rc = str(Seq(token).reverse_complement())
#     sym_pos = 23 - pos
#     # Canonical key: ordina tra le due coppie simmetriche
#     key1 = (token, pos)
#     key2 = (rc, sym_pos)
#     canonical_key = min(key1, key2)
#     merged_symmetric_scores[canonical_key] = merged_symmetric_scores.get(canonical_key, 0) + score

# # Ordina per score decrescente
# nomi_ordinati_pos, score_ordinati = zip(*sorted(merged_symmetric_scores.items(), key=lambda x: x[1], reverse=True))

# # Formatta nome token@posizione
# nomi_ordinati_str = [f"{tok}@{pos}" for tok, pos in nomi_ordinati_pos]

# # Visualizza i top token+pos più importanti
# topk = 10
# plt.figure(figsize=(12, 5))
# plt.bar(nomi_ordinati_str[:topk], score_ordinati[:topk], color='salmon')
# plt.xticks(rotation=90)
# plt.title(f"Top {topk} 6-mer@position (con simmetria RC + pos) per attenzione media")
# plt.xlabel("6-mer@position (canonical pair)")
# plt.ylabel("Attenzione media ricevuta")
# plt.tight_layout()
# plt.show()

# # Output testuale
# print(f"\nTop {topk} 6-mer@position con simmetria RC e posizione:")
# for i in range(topk):
#     print(f"{i+1:2d}. {nomi_ordinati_str[i]} → {score_ordinati[i]:.4f}")


In [ ]:
# #gemini SENZA POSIZIONE 

# import torch
# from collections import defaultdict
# import matplotlib.pyplot as plt

# # VERSIONE (SOLO TOKEN, POSIZIONE INDIFFERENTE)
# # ---------------------------------------------------

# # 7. Filtro per veri negativi (questa parte rimane invariata)
# data_reduced = [
#     row for row, p, tl in zip(dataset_test_to_study, preds_int, val_labels)
#     #if p == tl and tl == 0
# ]

# all_matrices_dir_filtered = [
#     m for m, p, tl in zip(all_matrices_dir, preds_int, val_labels)
#     #if p == tl and tl == 0
# ]

# all_matrices_rc_filtered = [
#     m for m, p, tl in zip(all_matrices_rc, preds_int, val_labels)
#     #if p == tl and tl == 0
# ]

# stacked_dir = torch.stack(all_matrices_dir_filtered)
# stacked_rc = torch.stack(all_matrices_rc_filtered)


# lista_dizionari = data_reduced
# attention_tensor_dir = stacked_dir
# attention_tensor_rc = stacked_rc


# def tokenize_dna_sequence(seq, k=6, max_tokens=24):
#     """Tokenizza una sequenza di DNA in k-mer non sovrapposti."""
#     return [seq[i:i+k] for i in range(0, k*max_tokens, k)]

# # Indici validi nel tensore di attenzione (escludendo CLS, EOS, etc.)
# start_idx = 1
# end_idx = 25
# valid_indices = list(range(start_idx, end_idx))  # da 1 a 24

# # Dizionari per accumulare i valori (rinominati per chiarezza)
# token_attention_totals = defaultdict(float) # <--- MODIFICA: Non più "tokenpos"
# token_counts = defaultdict(int)             # <--- MODIFICA: Non più "tokenpos"

# # 8. Calcolo dell'attenzione media per ogni TOKEN
# for i, sample in enumerate(lista_dizionari):
#     sequence = sample['sequence']
#     tokens = tokenize_dna_sequence(sequence)
#     # Attenzione ricevuta dal token CLS (indice 0) verso tutti gli altri token
#     attn_received = attention_tensor_dir[i][0, :] # Forma (28,)

#     for pos_in_tensor, pos_in_token_list in enumerate(valid_indices):
#         token = tokens[pos_in_tensor]
#         attn_value = attn_received[pos_in_token_list].item()
        
#         # La chiave ora è SOLO il token, la posizione viene ignorata.
#         key = token # <--- MODIFICA FONDAMENTALE
        
#         token_attention_totals[key] += attn_value
#         token_counts[key] += 1

# #RC
# for i, sample in enumerate(lista_dizionari):
#     sequence = str(Seq(sample['sequence']).reverse_complement())
#     tokens = tokenize_dna_sequence(sequence)
#     # Attenzione ricevuta dal token CLS (indice 0) verso tutti gli altri token
#     attn_received = attention_tensor_rc[i][0, :] # Forma (28,)

#     for pos_in_tensor, pos_in_token_list in enumerate(valid_indices):
#         token = tokens[pos_in_tensor]
#         attn_value = attn_received[pos_in_token_list].item()
        
#         # La chiave ora è SOLO il token, la posizione viene ignorata.
#         key = token # <--- MODIFICA FONDAMENTALE
        
#         token_attention_totals[key] += attn_value
#         token_counts[key] += 1

    
# # Calcola l'attenzione media per ciascun token unico
# token_mean_attention = {
#     k: token_attention_totals[k] / token_counts[k]
#     for k in token_attention_totals
# }

# s=0

# token_mean_attention_simmetric = {}
# for k,v in token_mean_attention.items():
#     k_rc = str(Seq(k).reverse_complement())
#     try:
#         if k != k_rc:
#             try:
#                 key_name = k+'\n' + k_rc
#                 token_mean_attention_simmetric[key_name]=token_mean_attention[k]+token_mean_attention[k_rc]
#             except:
#                 key_name = k
#                 token_mean_attention_simmetric[key_name]=token_mean_attention[k]                

#         else:
#             key_name = k 
#             token_mean_attention_simmetric[key_name]=token_mean_attention[k]
#     except:
        
#         s+=1

# total_keys = token_mean_attention_simmetric.keys()

# cleaned_dict={}

# for key, value in token_mean_attention_simmetric.items():
#     try:
#         k1, k2 = key.split('\n')
#         # ordina la coppia in modo deterministico
#         k_sorted = tuple(sorted([k1, k2]))
#         combined_key = '\n'.join(k_sorted)
        
#         # se la chiave combinata non è già presente, salvala
#         if combined_key not in cleaned_dict:
#             cleaned_dict[combined_key] = value
#     except:
#         cleaned_dict[key]=value
        

# # 9. Ordina i risultati per score decrescente
# sorted_items = sorted(cleaned_dict.items(), key=lambda item: item[1], reverse=True)
# # Ora 'nomi_ordinati' contiene direttamente i token (stringhe)
# nomi_ordinati, score_ordinati = zip(*sorted_items) # <--- MODIFICA: Unpacking più semplice

# # 10. Visualizza i risultati
# topk = 20
# plt.figure(figsize=(12, 5))
# # I nomi per le barre del grafico sono già pronti in 'nomi_ordinati'
# plt.bar(nomi_ordinati[:topk], score_ordinati[:topk], color='cornflowerblue') # <--- MODIFICA
# plt.xticks(rotation=90)
# # Aggiorna titolo e label degli assi
# plt.title(f"Mean Attention Weight of the Top {topk} 6-mers") # <--- MODIFICA
# plt.xlabel("6-mers") # <--- MODIFICA
# plt.ylabel("Mean Attention Weight")
# plt.hlines(y=np.mean(score_ordinati[:]),xmin=-1, xmax=20, colors='blue')
# plt.tight_layout()
# plt.show()

# # 11. Stampa i risultati testuali
# print(f"\nMean Attention weight of the Top {topk} 6-mers:") # <--- MODIFICA
# for i in range(topk):
#     # La stampa ora usa direttamente il token da 'nomi_ordinati'
#     print(f"{i+1:2d}. {nomi_ordinati[i]} → {score_ordinati[i]:.4f}") # <--- MODIFICA

In [ ]:
import torch
from collections import defaultdict
import matplotlib.pyplot as plt
import numpy as np
from Bio.Seq import Seq

# ----------------------
# PARAMETRI
# ----------------------
k = 6
max_tokens = 32#24
start_idx = 1
end_idx = 33#25
valid_indices = list(range(start_idx, end_idx))  # da 1 a 24

topk = 20

# ----------------------
# FUNZIONE TOKENIZER
# ----------------------
def tokenize_dna_sequence(seq, k=6, max_tokens=24):
    return [seq[i:i + k] for i in range(0, k * max_tokens, k)]

# ----------------------
# 1. FILTRA VERI NEGATIVI
# ----------------------
data_reduced = [
    row for row, p, tl in zip(dataset_test_to_study, preds_int, val_labels)
    #if p == tl and tl == 0
]
all_matrices_dir_filtered = [
    m for m, p, tl in zip(all_matrices_dir, preds_int, val_labels)
    #if p == tl and tl == 0
]
all_matrices_rc_filtered = [
    m for m, p, tl in zip(all_matrices_rc, preds_int, val_labels)
    #if p == tl and tl == 0
]

attention_tensor_dir = torch.stack(all_matrices_dir_filtered)
attention_tensor_rc = torch.stack(all_matrices_rc_filtered)

# ----------------------
# 2. CALCOLO ATTENZIONE MEDIA PER TOKEN
# ----------------------
token_attention_totals = defaultdict(float)
token_counts = defaultdict(int)

for is_rc, attention_tensor in [(False, attention_tensor_dir), (True, attention_tensor_rc)]:
    for i, sample in enumerate(data_reduced):
        seq = sample['sequence']
        if is_rc:
            seq = str(Seq(seq).reverse_complement())
        tokens = tokenize_dna_sequence(seq, k=k, max_tokens=max_tokens)
        attn_received = attention_tensor[i][0, :]  # CLS token attention to all others

        for pos_in_tensor, pos_in_token_list in enumerate(valid_indices):
            if pos_in_tensor >= len(tokens):
                break
            token = tokens[pos_in_tensor]
            attn_value = attn_received[pos_in_token_list].item()
            token_attention_totals[token] += attn_value
            token_counts[token] += 1

# ----------------------
# 3. ATTENZIONE MEDIA PER TOKEN
# ----------------------
token_mean_attention = {
    k.upper(): token_attention_totals[k] / token_counts[k]
    for k in token_attention_totals
}

# ----------------------
# 4. AGGREGAZIONE SIMMETRICA (TOKEN + RC)
# ----------------------
token_mean_attention_symmetric = {}
used_keys = set()

for k, v in token_mean_attention.items():
    if k in used_keys:
        continue
    k_rc = str(Seq(k).reverse_complement())
    if k_rc in token_mean_attention:
        if k != k_rc:
            key_combined = '\n'.join(sorted([k, k_rc]))
            total_score = token_mean_attention[k] + token_mean_attention[k_rc]
            used_keys.update([k, k_rc])
        else:
            key_combined = k
            total_score = token_mean_attention[k] 
            used_keys.add(k)
        token_mean_attention_symmetric[key_combined] = total_score
    else:
        token_mean_attention_symmetric[k] = v
        used_keys.add(k)

# ----------------------
# 5. ORDINAMENTO E PLOT
# ----------------------
sorted_items = sorted(token_mean_attention_symmetric.items(), key=lambda item: item[1], reverse=True)
nomi_ordinati, score_ordinati = zip(*sorted_items)

plt.figure(figsize=(12, 5))
plt.bar(nomi_ordinati[:topk], score_ordinati[:topk], color='cornflowerblue')
plt.xticks(rotation=90)
plt.title(f"Mean Attention Weight of the Top {topk} 6-mers")
plt.xlabel("6-mers")
plt.ylabel("Mean Attention Weight")
plt.hlines(y=np.mean(score_ordinati), xmin = 0, xmax = topk - 1, colors='blue', linestyles='dashed',label='Distribution Mean')
plt.hlines(y=np.mean(score_ordinati)+np.std(score_ordinati), xmin=0, xmax=topk - 1, colors='lightsteelblue', linestyles='dashed', label='Distribution Mean + std')
plt.legend()
plt.tight_layout()
plt.show()

# ----------------------
# 6. STAMPA RISULTATI
# ----------------------
print(f"\nMean Attention Weight of the Top {topk} 6-mers:")
for i in range(topk):
    print(f"{i + 1:2d}. {nomi_ordinati[i]} → {score_ordinati[i]:.4f}")


In [ ]:
torch.concatenate([attention_tensor,attention_tensor],axis=0).shape

In [ ]:
import torch
import matplotlib.pyplot as plt

TOKEN_START_IDX = 1
TOKEN_END_IDX = 33#25

#data_samples = data_reduced *2
attention_matrices_dir = all_matrices_dir_filtered
attention_matrices_rc = all_matrices_rc_filtered#[invert_attention(x) for x in all_matrices_rc_filtered]

try:
    attention_tensor_dir = torch.stack(attention_matrices_dir)
    attention_tensor_rc = torch.stack(attention_matrices_rc)
    
except (RuntimeError, TypeError):
    print("Errore: impossibile creare un tensore. Assicurati che tutte le matrici abbiano la stessa dimensione.")
    exit()


attention_from_cls_dir = attention_tensor_dir[:, 0, TOKEN_START_IDX:TOKEN_END_IDX]
attention_from_cls_rc =torch.flip(attention_tensor_rc[:, 0, TOKEN_START_IDX:TOKEN_END_IDX],dims = [1])

attention_from_cls= torch.concatenate([attention_from_cls_dir, attention_from_cls_rc],axis=0)

mean_attention_per_position = torch.mean(attention_from_cls, dim=0)

positions = list(range(TOKEN_START_IDX, TOKEN_END_IDX))
scores = mean_attention_per_position.tolist()
tick_labels = [f'{(pos-1)*6}-{pos*6}' for pos in positions]

plt.figure(figsize=(12, 5))
plt.bar(positions, scores, color='darkcyan', width=0.8)
plt.title("Mean Attention Weight for each Sequence Position", fontsize=16)
plt.xlabel("Sequence Position", fontsize=12)
plt.ylabel("Mean Attention Weight", fontsize=12)
plt.xticks(ticks=positions, labels=tick_labels, rotation=45, ha="right")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

print("\n--- Attenzione Media per Posizione ---")
for pos, score in zip(positions, scores):
    print(f"Posizione {pos:2d} (Intervallo {tick_labels[pos-1]:>7}) → {score:.4f}")

In [ ]:
# import torch
# import matplotlib.pyplot as plt
# from collections import defaultdict

# # Supposti input
# # - lista_dizionari: lista di dict, ciascuno con 'sequence': str (lunghezza almeno 6*24 = 144)
# # - attention_tensor: torch.Tensor di shape (N, 28, 28)

# data_reduced = [
#     row for row, p, tl in zip(dataset_test_to_study, preds_int, val_labels)
#     #if p == tl and tl == 0
# ]

# all_matrices_dir_filtered = [
#     m for m, p, tl in zip(all_matrices_dir, preds_int, val_labels)
#     #if p == tl and tl == 0
# ]

# all_matrices_rc_filtered = [
#     m for m, p, tl in zip(all_matrices_rc, preds_int, val_labels)
#     #if p == tl and tl == 0
# ]

# stacked_dir = torch.stack(all_matrices_dir_filtered)
# stacked_rc = torch.stack(all_matrices_rc_filtered)


# lista_dizionari = data_reduced  # oppure dataset_train
# attention_tensor_dir = stacked_dir
# attention_tensor_rc = stacked_rc

# # Parametri
# start_idx = 1      # posizione dopo il token [CLS]
# end_idx = 25       # esclude gli ultimi 3 token
# valid_indices = list(range(start_idx, end_idx))  # 1..24

# # Output: attenzione aggregata per posizione
# position_attention_totals = defaultdict(float)
# position_counts = defaultdict(int)

# # Itera su tutte le sequenze e matrici di attenzione
# for i, sample in enumerate(lista_dizionari):
#     # attenzione ricevuta da ciascun token (sommata lungo la dimensione delle query)
#     attn_received = attention_tensor_rc[i][0,:]   #.sum(dim=0)  # shape (28,)
    
#     for idx_in_tensor, pos in enumerate(valid_indices):  # pos va da 1 a 24
#         attn_value = attn_received[pos].item()
#         position_attention_totals[pos] += attn_value
#         position_counts[pos] += 1

# # Calcola media attenzione per ciascuna posizione
# position_mean_attention = {pos: position_attention_totals[pos] / position_counts[pos] for pos in valid_indices}

# # Ordina per posizione
# positions = sorted(position_mean_attention.keys())
# scores = [position_mean_attention[pos] for pos in positions]


# # positions_true = []
# # for i in positions:
# #     positions_true.append(f'{(i-1)*6}-{i*6}')


# # Visualizza
# plt.figure(figsize=(10, 4))
# plt.bar(positions, scores, color='mediumseagreen')
# plt.xticks(positions)
# plt.title("Attenzione media ricevuta per posizione nella sequenza (1–24)")
# plt.xlabel("Posizione del token nella sequenza")
# plt.ylabel("Attenzione media ricevuta")
# plt.tight_layout()
# plt.show()

# # Output testuale
# print("\nAttenzione media per posizione:")
# for pos in positions:
#     print(f"Posizione {pos:2d} → {position_mean_attention[pos]:.4f}")


In [ ]:
# import torch
# from collections import defaultdict
# import matplotlib.pyplot as plt
# from Bio.Seq import Seq


# class DNAAttentionAnalyzer:
#     """Analyzer for DNA sequence attention patterns in transformer models."""
    
#     def __init__(self, k=6, max_tokens=24, start_idx=1, end_idx=25):
#         self.k = k
#         self.max_tokens = max_tokens
#         self.start_idx = start_idx
#         self.end_idx = end_idx
#         self.valid_indices = list(range(start_idx, end_idx))
        
#         # Storage for attention calculations
#         self.tokenpos_attention_totals = defaultdict(float)
#         self.tokenpos_counts = defaultdict(int)
    
#     def tokenize_dna_sequence(self, seq):
#         """Tokenize a DNA sequence into non-overlapping k-mers."""
#         return [seq[i:i+self.k] for i in range(0, self.k * self.max_tokens, self.k)]
    
#     def filter_true_negatives(self, dataset, predictions, labels, attention_dir, attention_rc):
#         """Filter data to keep only true negatives (correct predictions where label=0)."""
#         mask = [(p == tl and tl == 0) for p, tl in zip(predictions, labels)]
        
#         filtered_data = [row for row, keep in zip(dataset, mask) if keep]
#         filtered_dir = [m for m, keep in zip(attention_dir, mask) if keep]
#         filtered_rc = [m for m, keep in zip(attention_rc, mask) if keep]
        
#         return filtered_data, torch.stack(filtered_dir), torch.stack(filtered_rc)
    
#     def process_attention_matrices(self, data, attention_tensor, use_reverse_complement=False):
#         """Process attention matrices to accumulate token-position attention scores."""
#         for i, sample in enumerate(data):
#             sequence = sample['sequence']
            
#             if use_reverse_complement:
#                 sequence = str(Seq(sequence).reverse_complement())
            
#             tokens = self.tokenize_dna_sequence(sequence)
            
#             # Attention received by CLS token (index 0) from all other tokens
#             attn_received = attention_tensor[i][0, :]
            
#             for pos_in_tensor, pos_in_token_list in enumerate(self.valid_indices):
#                 if pos_in_tensor < len(tokens):  # Safety check
#                     token = tokens[pos_in_tensor]
#                     attn_value = attn_received[pos_in_token_list].item()
#                     key = (token, pos_in_token_list)
                    
#                     self.tokenpos_attention_totals[key] += attn_value
#                     self.tokenpos_counts[key] += 1
    
#     def calculate_mean_attention(self):
#         """Calculate mean attention scores for each token-position pair."""
#         return {
#             k: self.tokenpos_attention_totals[k] / self.tokenpos_counts[k]
#             for k in self.tokenpos_attention_totals
#         }
    
#     def get_top_results(self, mean_attention, top_k=10):
#         """Get top-k results sorted by attention score."""
#         sorted_items = sorted(mean_attention.items(), key=lambda x: x[1], reverse=True)
#         token_pos_pairs, scores = zip(*sorted_items)
        
#         # Format as "token@position" for display
#         formatted_names = [f"{tok}@{pos}" for tok, pos in token_pos_pairs]
        
#         return formatted_names[:top_k], scores[:top_k]
    
#     def visualize_results(self, names, scores, top_k=10):
#         """Create bar plot of top results."""
#         plt.figure(figsize=(12, 5))
#         plt.bar(names, scores, color='cornflowerblue')
#         plt.xticks(rotation=90)
#         plt.title(f"Top {top_k} 6-mer@position by Mean Attention")
#         plt.xlabel("6-mer@position")
#         plt.ylabel("Mean Attention Received")
#         plt.tight_layout()
#         plt.show()
    
#     def print_results(self, names, scores, top_k=10):
#         """Print formatted results."""
#         print(f"\nTop {top_k} 6-mer@position by mean attention:")
#         for i in range(min(top_k, len(names))):
#             print(f"{i+1:2d}. {names[i]} → {scores[i]:.4f}")
    
#     def analyze(self, dataset, predictions, labels, attention_dir, attention_rc, top_k=10):
#         """Complete analysis pipeline."""
#         # Filter for true negatives
#         filtered_data, stacked_dir, stacked_rc = self.filter_true_negatives(
#             dataset, predictions, labels, attention_dir, attention_rc
#         )
        
#         # Process both direct and reverse complement attention
#         self.process_attention_matrices(filtered_data, stacked_dir, use_reverse_complement=False)
#         self.process_attention_matrices(filtered_data, stacked_rc, use_reverse_complement=True)
        
#         # Calculate results
#         mean_attention = self.calculate_mean_attention()
#         top_names, top_scores = self.get_top_results(mean_attention, top_k)
        
#         # Display results
#         self.visualize_results(top_names, top_scores, top_k)
#         self.print_results(top_names, top_scores, top_k)
        
#         return mean_attention, top_names, top_scores



# # Initialize analyzer
# analyzer = DNAAttentionAnalyzer(k=6, max_tokens=24)

# # Run complete analysis
# # Replace these with your actual data variables
# results = analyzer.analyze(
#     dataset_test_to_study, 
#     preds_int, 
#     val_labels, 
#     all_matrices_dir, 
#     all_matrices_rc,
#     top_k=10
# )

# # For the original inline approach (if you prefer):
# # mean_attention, top_names, top_scores = results

In [ ]:
import torch
from collections import defaultdict
import matplotlib.pyplot as plt
from Bio.Seq import Seq  # Make sure BioPython is imported for reverse_complement

# Filter for true negatives (prediction == label == 0)
def filter_true_negatives(data, preds, labels):
    return [item for item, p, tl in zip(data, preds, labels) if p == tl]# == 0]

data_reduced = filter_true_negatives(dataset_test_to_study, preds_int, val_labels)
all_matrices_dir_filtered = filter_true_negatives(all_matrices_dir, preds_int, val_labels)
all_matrices_rc_filtered = filter_true_negatives(all_matrices_rc, preds_int, val_labels)

# Stack filtered matrices into tensors
stacked_dir = torch.stack(all_matrices_dir_filtered)
stacked_rc = torch.stack(all_matrices_rc_filtered)

# Tokenize DNA sequence into non-overlapping k-mers
def tokenize_dna_sequence(seq, k=6, max_tokens=32):#24):
    return [seq[i:i+k] for i in range(0, k * max_tokens, k)]

# Compute average attention per (token, position)
def compute_attention_avg(data, attention_tensor, reverse_complement=False):
    attention_totals = defaultdict(float)
    counts = defaultdict(int)
    
    valid_indices = range(1, 33)#25)  # indices 1 to 24 (exclude CLS token 0)
    
    for i, sample in enumerate(data):
        sequence = sample['sequence']
        if reverse_complement:
            sequence = str(Seq(sequence).reverse_complement())
        
        tokens = tokenize_dna_sequence(sequence)
        attn_received = attention_tensor[i][0, :]  # Attention from CLS token
        
        for pos_in_tensor, pos_in_token_list in enumerate(valid_indices):
            token = tokens[pos_in_tensor]
            attn_value = attn_received[pos_in_token_list].item()
            key = (token, pos_in_token_list)
            
            attention_totals[key] += attn_value
            counts[key] += 1
            
    return attention_totals, counts

# Compute totals and counts for direct and reverse complement attention
totals_dir, counts_dir = compute_attention_avg(data_reduced, stacked_dir, reverse_complement=False)
totals_rc, counts_rc = compute_attention_avg(data_reduced, stacked_rc, reverse_complement=True)

# Combine totals and counts
for key, val in totals_rc.items():
    totals_dir[key] += val
    counts_dir[key] += counts_rc[key]

# Calculate mean attention per (token, position)
tokenpos_mean_attention = {
    key: totals_dir[key] / counts_dir[key] for key in totals_dir
}

# Sort descending by mean attention score
sorted_items = sorted(tokenpos_mean_attention.items(), key=lambda x: x[1], reverse=True)
tokens_positions, scores = zip(*sorted_items)
tokens_positions_str = [f"{token}, p={pos*6 - 5}-{pos*6}" for token, pos in tokens_positions]

# Plot top-k attention scores
topk = 20
plt.figure(figsize=(12, 5))
plt.bar(tokens_positions_str[:topk], scores[:topk], color='cornflowerblue')
plt.xticks(rotation=90)
plt.hlines(y=np.mean(scores), xmin = 0, xmax = topk - 1, colors='blue', linestyles='dashed',label='Distribution Mean')
plt.hlines(y=np.mean(scores)+np.std(scores), xmin=0, xmax=topk - 1, colors='lightsteelblue', linestyles='dashed', label='Distribution Mean + std')
plt.legend()



plt.title(f"Mean Attention Weight for Top {topk} 6-mer + position", fontsize=16)
plt.xlabel("6-mer + Sequence Position", fontsize=12)
plt.ylabel("Mean Attention Weight", fontsize=12)
plt.tight_layout()
plt.show()

# Print top-k textual results
print(f"\nTop {topk} 6-mer@posizione per attenzione media:")
for i in range(topk):
    print(f"{i+1:2d}. {tokens_positions_str[i]} → {scores[i]:.4f}")


In [ ]:
import torch
from collections import defaultdict
import matplotlib.pyplot as plt
from Bio.Seq import Seq  # Assicurati di avere Biopython

# Filtra i true negatives (prediction == label == 0)
def filter_true_negatives(data, preds, labels):
    return [item for item, p, tl in zip(data, preds, labels)if p == tl == 1]

data_reduced = filter_true_negatives(dataset_test_to_study, preds_int, val_labels)
all_matrices_dir_filtered = filter_true_negatives(all_matrices_dir, preds_int, val_labels)
all_matrices_rc_filtered = filter_true_negatives(all_matrices_rc, preds_int, val_labels)

# Stack delle matrici in tensori
stacked_dir = torch.stack(all_matrices_dir_filtered)
stacked_rc = torch.stack(all_matrices_rc_filtered)

# Tokenizza la sequenza DNA in k-mers
def tokenize_dna_sequence(seq, k=6, max_tokens=32):#24):
    return [seq[i:i+k] for i in range(0, k * max_tokens, k)]

# # Mappa la posizione in una delle 3 regioni
# def map_position_to_region(pos):
#     if 1 <= pos <= 8:
#         return 'sinistra'
#     elif 9 <= pos <= 16:
#         return 'centrale'
#     elif 17 <= pos <= 24:
#         return 'destra'
#     else:
#         return None  # Non dovrebbe mai capitare



# Mappa la posizione in una delle 3 regioni
def map_position_to_region(pos):
    if 9 <= pos <= 16:
        return 'centrale'
    else:
        return 'laterale'


# Calcola l'attenzione media aggregata per (token, regione)
def compute_attention_avg(data, attention_tensor, reverse_complement=False):
    attention_totals = defaultdict(float)
    counts = defaultdict(int)
    
    valid_indices = range(1, 33)#25)  # da 1 a 24 (escludendo il CLS token 0)
    
    for i, sample in enumerate(data):
        sequence = sample['sequence']
        if reverse_complement:
            sequence = str(Seq(sequence).reverse_complement())
        
        tokens = tokenize_dna_sequence(sequence)
        attn_received = attention_tensor[i][0, :]  # Attenzione dal CLS token

        for pos_in_tensor, pos_in_token_list in enumerate(valid_indices):
            token = tokens[pos_in_tensor]
            attn_value = attn_received[pos_in_token_list].item()
            region = map_position_to_region(pos_in_token_list)
            key = (token, region)
            
            attention_totals[key] += attn_value
            counts[key] += 1
            
    return attention_totals, counts

# Calcola totali e conteggi
totals_dir, counts_dir = compute_attention_avg(data_reduced, stacked_dir, reverse_complement=False)
totals_rc, counts_rc = compute_attention_avg(data_reduced, stacked_rc, reverse_complement=True)

# Somma le attenzioni dirette e reverse complement
for key, val in totals_rc.items():
    totals_dir[key] += val
    counts_dir[key] += counts_rc[key]

# Calcola attenzione media
token_region_mean_attention = {
    key: totals_dir[key] / counts_dir[key] for key in totals_dir
}

# Ordina in base all'attenzione media decrescente
sorted_items = sorted(token_region_mean_attention.items(), key=lambda x: x[1], reverse=True)
tokens_regions, scores = zip(*sorted_items)
tokens_regions_str = [f"{token}@{region}" for token, region in tokens_regions]

# Plot
topk = 20
plt.figure(figsize=(12, 5))
plt.bar(tokens_regions_str[:topk], scores[:topk], color='cornflowerblue')
plt.xticks(rotation=90)
plt.title(f"Top {topk} 6-mer@regione per attenzione media")
plt.xlabel("6-mer@regione")
plt.ylabel("Attenzione media ricevuta")
plt.tight_layout()
plt.show()

# Stampa testuale
print(f"\nTop {topk} 6-mer@regione per attenzione media:")
for i in range(topk):
    print(f"{i+1:2d}. {tokens_regions_str[i]} → {scores[i]:.4f}")


In [ ]:
#GEMINI --------------------

import torch
from collections import defaultdict
import matplotlib.pyplot as plt

# VERSIONE (TOKEN, POSIZIONE)
# 7. Filtro per veri negativi
data_reduced = [
    row for row, p, tl in zip(dataset_test_to_study, preds_int, val_labels)
    if p == tl and tl == 1
][2:]

all_matrices_dir_filtered = [
    m for m, p, tl in zip(all_matrices_dir, preds_int, val_labels)
    if p == tl and tl == 1
][2:]
all_matrices_rc_filtered = [
    m for m, p, tl in zip(all_matrices_rc, preds_int, val_labels)
    if p == tl and tl == 1
][2:]


stacked_dir = torch.stack(all_matrices_dir_filtered)
stacked_rc = torch.stack(all_matrices_rc_filtered)

lista_dizionari = [data_reduced[1]]
attention_tensor_dir = [stacked_dir[1]]
attention_tensor_rc = [stacked_rc[1]]


def tokenize_dna_sequence(seq, k=6, max_tokens=32):#24):
    """Tokenizza una sequenza di DNA in k-mer non sovrapposti."""
    return [seq[i:i+k] for i in range(0, k*max_tokens, k)]

# Indici validi nel tensore di attenzione (escludendo CLS, EOS, etc.)
start_idx = 1
end_idx = 33#25
valid_indices = list(range(start_idx, end_idx))  # da 1 a 24

# Dizionari per accumulare i valori
tokenpos_attention_totals = defaultdict(float)
tokenpos_counts = defaultdict(int)

# # 8. Calcolo dell'attenzione media per ogni (token, posizione)
# for i, sample in enumerate(lista_dizionari):
#     sequence = sample['sequence']
#     tokens = tokenize_dna_sequence(sequence)  # 24 token
#     # Attenzione ricevuta dal token CLS (indice 0) verso tutti gli altri token
#     attn_received = attention_tensor_dir[i][0, :] # Forma (28,)

#     for pos_in_tensor, pos_in_token_list in enumerate(valid_indices):
#         token = tokens[pos_in_tensor]
#         attn_value = attn_received[pos_in_token_list].item()
#         key = (token, pos_in_token_list) # La chiave ora è (token, posizione assoluta nel tensore)
        
#         tokenpos_attention_totals[key] += attn_value
#         tokenpos_counts[key] += 1


# 8. Calcolo dell'attenzione media per ogni (token, posizione)
for i, sample in enumerate(lista_dizionari):
    sequence = str(Seq(sample['sequence']).reverse_complement())
    tokens = tokenize_dna_sequence(sequence)  # 24 token
    # Attenzione ricevuta dal token CLS (indice 0) verso tutti gli altri token
    attn_received = attention_tensor_rc[i][0, :] # Forma (28,)

    for pos_in_tensor, pos_in_token_list in enumerate(valid_indices):
        token = tokens[pos_in_tensor]
        attn_value = attn_received[pos_in_token_list].item()
        key = (token, pos_in_token_list) # La chiave ora è (token, posizione assoluta nel tensore)
        
        tokenpos_attention_totals[key] += attn_value
        tokenpos_counts[key] += 1
        

# Calcola l'attenzione media
tokenpos_mean_attention = {
    k: tokenpos_attention_totals[k] / tokenpos_counts[k]
    for k in tokenpos_attention_totals
}

# 9. Ordina i risultati per score decrescente
# Si ordina direttamente dal dizionario delle medie, senza fusione con RC
sorted_items = sorted(tokenpos_mean_attention.items(), key=lambda x: x[1], reverse=True)
nomi_ordinati_pos, score_ordinati = zip(*sorted_items)

# Formatta i nomi come "token@posizione" per il grafico
nomi_ordinati_str = [f"{tok}@{pos}" for tok, pos in nomi_ordinati_pos]

# 10. Visualizza i risultati
topk = 24
plt.figure(figsize=(12, 5))
plt.bar(nomi_ordinati_str[:topk], score_ordinati[:topk], color='cornflowerblue')
plt.xticks(rotation=90)
plt.title(f"Top {topk} 6-mer@posizione per attenzione media")
plt.xlabel("6-mer@posizione")
plt.ylabel("Attenzione media ricevuta")
plt.tight_layout()
plt.show()

# 11. Stampa i risultati testuali
print(f"\nTop {topk} 6-mer@posizione per attenzione media:")
for i in range(topk):
    print(f"{i+1:2d}. {nomi_ordinati_str[i]} → {score_ordinati[i]:.4f}")

In [ ]:
# # -------------------------------
# # 3. Uso di DeepLiftSHAP
# # -------------------------------
# # Creiamo un interprete DeepLiftShap
# dl_shap = DeepLiftShap(model)

# # Input da spiegare (nuovi dati)
# inputs = data_reduced

# # Scelta delle baseline: tipicamente 0 o input medi
# baselines = data_reduced*0

# # Calcolo delle attribuzioni
# attributions = dl_shap.attribute(inputs, baselines=baselines)

# print("Input:\n", inputs)
# print("Attribuzioni DeepLiftSHAP:\n", attributions)


In [ ]:
import matplotlib.pyplot as plt

# Supponiamo che tokenpos_mean_attention sia già definito
# È un dizionario tipo: {('TGGTAG', 1): 0.025, ...}


positions = [p for (kmer, p) in tokenpos_mean_attention.keys()]
values = [v for (kmer, p), v in tokenpos_mean_attention.items()]
kmers = [kmer.upper() for (kmer, p) in tokenpos_mean_attention.keys()]

# Calcolo font size dinamico
min_size = 8
max_size = 20
min_v = min(values)
max_v = max(values)
norm = lambda v: min_size + (v - min_v) / (max_v - min_v + 1e-8) * (max_size - min_size)

# Plot
plt.figure(figsize=(15, 8))
plt.bar(positions, values, color='cornflowerblue')

# Etichette con dimensione in base a valore
for x, y, kmer in zip(positions, values, kmers):
    plt.text(x, y + 0.001, kmer, ha='center', va='bottom', fontsize=norm(y), rotation=0)

plt.xlabel("Token position")
plt.title("Example of single seqence sequence analysis")
plt.ylabel("Attention Weight", fontsize=12)
plt.xticks(positions)  # Assicura che tutte le posizioni siano mostrate
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Dati

# Normalizzazione font size
min_size = 8
max_size = 28
min_val = min(values)
max_val = max(values)
scale = lambda v: min_size + (v - min_val) / (max_val - min_val + 1e-8) * (max_size - min_size)

# Plot
plt.figure(figsize=(len(kmers) * 1.2, 2))
x_pos = 0  # posizione orizzontale cumulativa

for kmer, val in zip(kmers, values):
    fs = scale(val)
    plt.text(x_pos, 0, kmer, fontsize=fs, ha='left', va='center', family='monospace')
    # Avanza la posizione: più grande è il font, più spazio serve
    x_pos += len(kmer) * (fs * 0.001)  # tweak questo coefficiente per densità

plt.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
import copy

dataset = copy.deepcopy([i for i in dataset_sapiens])
# 1. Filtra dataset con sequenze lunghezza 28
#dataset = [i for i in dataset_sapiens].copy() #if i['embedding'].shape[0] == 28]

# 2. Seleziona dati di test
dataset_test_to_study = [dataset[i] for i in internal_val_idx].copy()  # Assicurati che gli indici siano corretti


for i in range(len(dataset_test_to_study)):
    dataset_test_to_study[i]['embedding'][6:7,:]=0
    dataset_test_to_study[i]['embedding_rev'][18:19,:]=0
    
# 3. Crea dataloader
test = Nuc_Dataset(dataset_test_to_study, dim_embedding)
test = DataLoader(test, batch_size=64, shuffle=False)

# 4. Inference
metrics, val_labels, val_preds, importance, importance_rc = test_classification(model_internal, test)

# 5. Preds binarie
preds_int = [p > 0.5 for p in val_preds]

# 6. Estrai matrici direzionali
all_matrices_dir = [m for batch in importance for m in batch]  # importance = [B, 28, 28]
all_matrices_rc = [m for batch in importance_rc for m in batch]

all_matrices_tot = all_matrices_dir + all_matrices_rc#[x + invert_attention(y) for x,y in zip(all_matrices_dir,all_matrices_rc)]

preds_int = preds_int + preds_int

dataset_test_to_study *= 2

# 7. Etichette originali
true_label = val_labels + val_labels

# 8. Filtro per veri negativi
all_matrices_tot_filtered = [
    m for m, p, tl in zip(all_matrices_tot, preds_int, true_label)
    #if p == 0 and tl == 0 and m.shape == (28, 28)
]

# 9. Calcolo media e visualizzazione
if len(all_matrices_tot_filtered) > 0:
    stacked = torch.stack(all_matrices_tot_filtered)
    media = stacked[:,0,:].mean(dim=0).unsqueeze(0)
    media_norm = (media - media.min()) / (media.max() - media.min() + 1e-8)

    plt.figure(figsize=(6, 5))
    sns.heatmap(media_norm.cpu().numpy(), cmap="viridis")
    plt.title("Heatmap normalizzata - Veri negativi (p=0, tl=0)")
    plt.show()
else:
    print("⚠️ Nessun vero negativo trovato con shape [28, 28].")


In [ ]:
val_preds#0.00065244175

# [0.99641716,
#  0.9750645,
#  0.016087318,
#  0.8559031,
#  0.89552456,
#  0.018351797,
#  0.0070057027,
#  0.0021655096,
#  0.019678038,
#  0.7941537,
#  0.9603779,
#  0.0006276016,

In [ ]:
#FINEEEEEEEEEEE

In [ ]:
# #VERSIONE (TOKEN,POSIZIONE)
# data_reduced = [
#     row for row, p, tl in zip(dataset_test_to_study, preds_int, val_labels)
#     if p == tl and tl == 0 #if p == 0 and tl == 0
# ]


# # 8. Filtro per veri negativi
# all_matrices_tot_filtered = [
#     m for m, p, tl in zip(all_matrices_tot, preds_int, val_labels)
#     if p == tl and tl == 0 #if p == 0 and tl == 0 and m.shape == (28, 28)
# ]
# stacked = torch.stack(all_matrices_tot_filtered)

# lista_dizionari = [data_reduced[0]]
# attention_tensor = [stacked[0]]

# def tokenize_dna_sequence(seq, k=6, max_tokens=24):
#     return [seq[i:i+k] for i in range(0, k*max_tokens, k)]

# start_idx = 1
# end_idx = 25
# valid_indices = list(range(start_idx, end_idx))  # 1..24

# tokenpos_attention_totals = defaultdict(float)
# tokenpos_counts = defaultdict(int)

# for i, sample in enumerate(lista_dizionari):
#     sequence = sample['sequence']
#     tokens = tokenize_dna_sequence(sequence)  # 24 token
#     attn_received = attention_tensor[i][0,:]   #.sum(dim=0)  # shape (28,)

#     for pos_in_tensor, pos_in_token in enumerate(valid_indices):
#         token = tokens[pos_in_tensor]
#         attn_value = attn_received[pos_in_token].item()
#         key = (token, pos_in_token)
#         tokenpos_attention_totals[key] += attn_value
#         tokenpos_counts[key] += 1

# # Calcola attenzione media per (token, posizione)
# tokenpos_mean_attention = {
#     k: tokenpos_attention_totals[k] / tokenpos_counts[k]
#     for k in tokenpos_attention_totals
# }

# # Combina simmetricamente: (token, pos) ↔ (RC(token), 23 - pos)
# merged_symmetric_scores = {}

# for (token, pos), score in tokenpos_mean_attention.items():
#     rc = str(Seq(token).reverse_complement())
#     sym_pos = 23 - pos
#     # Canonical key: ordina tra le due coppie simmetriche
#     key1 = (token, pos)
#     key2 = (rc, sym_pos)
#     canonical_key = min(key1, key2)
#     merged_symmetric_scores[canonical_key] = merged_symmetric_scores.get(canonical_key, 0) + score

# # Ordina per score decrescente
# nomi_ordinati_pos, score_ordinati = zip(*sorted(merged_symmetric_scores.items(), key=lambda x: x[1], reverse=True))

# # Formatta nome token@posizione
# nomi_ordinati_str = [f"{tok}@{pos}" for tok, pos in nomi_ordinati_pos]

# # Visualizza i top token+pos più importanti
# topk = 24
# plt.figure(figsize=(12, 5))
# plt.bar(nomi_ordinati_str[:topk], score_ordinati[:topk], color='salmon')
# plt.xticks(rotation=90)
# plt.title(f"Top {topk} 6-mer@position (con simmetria RC + pos) per attenzione media")
# plt.xlabel("6-mer@position (canonical pair)")
# plt.ylabel("Attenzione media ricevuta")
# plt.tight_layout()
# plt.show()

# # Output testuale
# print(f"\nTop {topk} 6-mer@position con simmetria RC e posizione:")
# for i in range(topk):
#     print(f"{i+1:2d}. {nomi_ordinati_str[i]} → {score_ordinati[i]:.4f}")


In [ ]:
sequence

In [ ]:
len('TTTTTTTTTTTTTTTTTTTTTTTTTT')/6

In [ ]:
#versione senza posizione

# - lista_dizionari: lista di dict, ciascuno con 'sequence': str (lunghezza almeno 6*24 = 144)
# - attention_tensor: torch.Tensor di shape (N, 28, 28)

lista_dizionari = data_reduced #dataset_train
attention_tensor = stacked

def tokenize_dna_sequence(seq, k=6, max_tokens=24):
    """Restituisce una lista di 24 token 6-mer non sovrapposti."""
    return [seq[i:i+k] for i in range(0, k*max_tokens, k)]

# Parametri
start_idx = 1      # posizione dopo il token [CLS]
end_idx = 25       # esclude gli ultimi 3 token
valid_indices = list(range(start_idx, end_idx))  # 1..24

# Output: attenzione aggregata per 6-mer
token_attention_totals = defaultdict(float)
token_counts = defaultdict(int)

# Itera su tutte le sequenze e matrici di attenzione
for i, sample in enumerate(lista_dizionari):
    sequence = sample['sequence']
    tokens = tokenize_dna_sequence(sequence)  # 24 token

    # attenzione ricevuta da ciascun token (sommata lungo la dimensione delle query)
    attn_received = attention_tensor[i][0,:]   #.sum(dim=0)  # shape (28,)
    for pos_in_tensor, pos_in_token in enumerate(valid_indices):
        token = tokens[pos_in_tensor]  # corrisponde a posizione 1..24
        attn_value = attn_received[pos_in_token].item()
        token_attention_totals[token] += attn_value
        token_counts[token] += 1

# Calcola media attenzione per 6-mer
token_mean_attention = {tok: token_attention_totals[tok] / token_counts[tok] for tok in token_attention_totals}

# Ordina per attenzione media
sorted_tokens = sorted(token_mean_attention.items(), key=lambda x: x[1], reverse=True)

# Visualizza i top 20 token più importanti
topk = 5000
top_tokens, top_scores = zip(*sorted_tokens[:topk])



######  SOMMO I VALORI DELLA SEQ E RC SEQ  #######

# Dizionario per sommare punteggi di token e RC
merged_scores = {}

for token, score in zip(top_tokens, top_scores):
    rc = str(Seq(token).reverse_complement())
    key = min(token, rc)  # usa la versione lessicograficamente minore per evitare duplicati
    merged_scores[key] = merged_scores.get(key, 0) + score

# Ordina per score decrescente
nomi_ordinati, score_ordinati = zip(*sorted(merged_scores.items(), key=lambda x: x[1], reverse=True))

# Se vuoi convertirli in liste
nomi_ordinati = list(nomi_ordinati)
score_ordinati = list(score_ordinati)

# Visualizza i top 10 token più importanti
topk = 20
plt.figure(figsize=(12, 5))
plt.bar(nomi_ordinati[:topk], score_ordinati[:topk], color='salmon')
plt.xticks(rotation=90)
plt.title(f"Top {topk} 6-mer (token + RC) per attenzione media ricevuta")
plt.xlabel("6-mer (canonical form)")
plt.ylabel("Attenzione media ricevuta (somma con RC)")
plt.tight_layout()
plt.show()

# Output testuale
print(f"\nTop {topk} token 6-mer più importanti (RC combined):")
for i in range(topk):
    print(f"{i+1:2d}. {nomi_ordinati[i]} → {score_ordinati[i]:.4f}")


In [ ]:
from Bio.Seq import Seq    

tt = list(top_tokens)
count =0
for i in tt:
    try:
        RC = str(Seq(i).reverse_complement())
        _ = tt.index(RC)
        # print(tt.index(RC))
        # print(i)
        count+=1
    except:
        continue
print(count)

In [ ]:
import torch
import matplotlib.pyplot as plt
from collections import defaultdict

# Supposti input
# - lista_dizionari: lista di dict, ciascuno con 'sequence': str (lunghezza almeno 6*24 = 144)
# - attention_tensor: torch.Tensor di shape (N, 28, 28)

lista_dizionari = data_reduced  # oppure dataset_train
attention_tensor = stacked

# Parametri
start_idx = 1      # posizione dopo il token [CLS]
end_idx = 25       # esclude gli ultimi 3 token
valid_indices = list(range(start_idx, end_idx))  # 1..24

# Output: attenzione aggregata per posizione
position_attention_totals = defaultdict(float)
position_counts = defaultdict(int)

# Itera su tutte le sequenze e matrici di attenzione
for i, sample in enumerate(lista_dizionari):
    # attenzione ricevuta da ciascun token (sommata lungo la dimensione delle query)
    attn_received = attention_tensor[i][0,:]   #.sum(dim=0)  # shape (28,)
    
    for idx_in_tensor, pos in enumerate(valid_indices):  # pos va da 1 a 24
        attn_value = attn_received[pos].item()
        position_attention_totals[pos] += attn_value
        position_counts[pos] += 1

# Calcola media attenzione per ciascuna posizione
position_mean_attention = {pos: position_attention_totals[pos] / position_counts[pos] for pos in valid_indices}

# Ordina per posizione
positions = sorted(position_mean_attention.keys())
scores = [position_mean_attention[pos] for pos in positions]

# Visualizza
plt.figure(figsize=(10, 4))
plt.bar(positions, scores, color='mediumseagreen')
plt.xticks(positions)
plt.title("Attenzione media ricevuta per posizione nella sequenza (1–24)")
plt.xlabel("Posizione del token nella sequenza")
plt.ylabel("Attenzione media ricevuta")
plt.tight_layout()
plt.show()

# Output testuale
print("\nAttenzione media per posizione:")
for pos in positions:
    print(f"Posizione {pos:2d} → {position_mean_attention[pos]:.4f}")


In [ ]:
# Visualizza i top 20 token più importanti
topk = 100
top_tokens, top_scores = zip(*sorted_tokens[:topk])

sorted_scores = sorted(top_scores, reverse=True)

plt.figure(figsize=(10, 5))
plt.plot(sorted_scores, color='darkorange')
plt.title("Attenzione media ricevuta dai token 6-mer (ordinati)")
plt.xlabel("Token (ordinati per attenzione decrescente)")
plt.ylabel("Attenzione media ricevuta")
plt.tight_layout()
plt.show()


In [ ]:
import torch
import matplotlib.pyplot as plt
from collections import defaultdict

# Supposti input
# - lista_dizionari: lista di dict, ciascuno con 'sequence': str (lunghezza almeno 6*24 = 144)
# - attention_tensor: torch.Tensor di shape (N, 28, 28)

def tokenize_dna_sequence(seq, k=6, max_tokens=24):
    """Restituisce una lista di 24 token 6-mer non sovrapposti."""
    return [seq[i:i+k] for i in range(0, k*max_tokens, k)]

# Parametri
start_idx = 1
end_idx = 25  # esclude gli ultimi 3 token
valid_indices = list(range(start_idx, end_idx))  # 1..24
num_tokens = end_idx - start_idx  # 24

# Output: attenzione aggregata per coppie di token
pair_attention_totals = defaultdict(float)
pair_counts = defaultdict(int)

for i, sample in enumerate(lista_dizionari):
    sequence = sample['sequence']
    tokens = tokenize_dna_sequence(sequence)  # 24 token

    attn = attention_tensor[i]  # shape (28, 28)

    for qi, q_pos in enumerate(valid_indices):      # posizione query
        query_token = tokens[qi]
        for ki, k_pos in enumerate(valid_indices):  # posizione key
            key_token = tokens[ki]
            attn_value = attn[q_pos, k_pos].item()
            pair = (query_token, key_token)
            pair_attention_totals[pair] += attn_value
            pair_counts[pair] += 1

# Calcola media attenzione per coppie
pair_mean_attention = {
    pair: pair_attention_totals[pair] / pair_counts[pair]
    for pair in pair_attention_totals
}

# Ordina per attenzione media più alta
sorted_pairs = sorted(pair_mean_attention.items(), key=lambda x: x[1], reverse=True)

# Visualizza i top 50
topk = 20
top_pairs, top_scores = zip(*sorted_pairs[:topk])
pair_labels = [f"{p[0]}→{p[1]}" for p in top_pairs]

plt.figure(figsize=(14, 6))
plt.bar(pair_labels, top_scores, color='skyblue')
plt.xticks(rotation=90)
plt.title(f"Top {topk} coppie di token (6-mer) per attenzione media")
plt.xlabel("Query → Key")
plt.ylabel("Attenzione media")
plt.tight_layout()
plt.show()

# Output testuale
print(f"\nTop {topk} coppie di token più importanti:")
for i in range(topk):
    print(f"{i+1:2d}. {top_pairs[i][0]} → {top_pairs[i][1]} → {top_scores[i]:.4f}")


from sklearn.metrics import confusion_matrix
confusion_matrix(true_label,preds)


In [ ]:
positive_validation =  [i for i in internal_val_subset if i['label'] == 1 ]
negative_validation =   [i for i in internal_val_subset if i['label'] == 0 ]


In [ ]:
d_true = positive_validation
test = Nuc_Dataset(d_true, dim_embedding)

test_nup2 = DataLoader(test, batch_size=64, shuffle=True, collate_fn=collate_fn)
a,b,c,true,pred = test_classification(model_internal,test_nup2)
import seaborn as sns


#true_pred
b = [b for b,p in zip(b,pred) if p==1]

somma = sum(t.cpu().sum(dim=0) for t in b)
sns.heatmap(torch.mean(somma,dim=0).unsqueeze(0)/30)

In [ ]:
d_true =negative_validation
test = Nuc_Dataset(d_true, dim_embedding)

test_nup2 = DataLoader(test, batch_size=64, shuffle=True, collate_fn=collate_fn)
a,b,c,true,pred = test_classification(model_internal,test_nup2)
import seaborn as sns

#true_pred
b = [b for b,p in zip(b,pred) if p==0]

somma = sum(t.cpu().sum(dim=0) for t in b)
sns.heatmap(torch.mean(somma,dim=0).unsqueeze(0)/30)

In [ ]:
torch.mean(somma,dim=0).unsqueeze(0).squeeze()[5]

In [ ]:
###################

In [ ]:
# #CV WITH EARLY STOPPING
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# from sklearn.model_selection import StratifiedKFold, train_test_split
# from torch.utils.data import Subset, DataLoader
# import numpy as np
# import torch

# dim_embedding = 2560#2560  # o 768 a seconda del tuo caso
# dataset_train = dataset_melanogasters#[i for i in dataset_HS_PR if i['embedding'].shape[0] ==28]
# dataset_train = Nuc_Dataset(dataset_train, dim_embedding,drop_last=False)
# labels = np.array([dataset_train[i]['label'].item() for i in range(len(dataset_train))])

# skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# fold_results = []

# for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
#     print(f"\n--- Fold {fold+1} ---")

#     # Divisione fold in train/val
#     train_labels = labels[train_idx]
    
#     # Split interno train/test per selezione epoche
#     internal_train_idx, internal_val_idx = train_test_split(
#         train_idx,
#         test_size=0.05,
#         stratify=train_labels,
#         random_state=fold  # diverso per ogni fold
#     )

#     internal_train_subset = Subset(dataset_train, internal_train_idx)
#     internal_val_subset = Subset(dataset_train, internal_val_idx)
    
#     dataloader_internal_train = DataLoader(internal_train_subset, batch_size=64, shuffle=True)#, collate_fn=collate_fn)
#     dataloader_internal_val = DataLoader(internal_val_subset, batch_size=64, shuffle=False)#, collate_fn=collate_fn)

#     # Modello + Pesi
#     transf_parameters_cls = {'input_dim': dim_embedding, 'dropout_rate': 0.}
#     transf_parameters_att = {'input_dim': dim_embedding, 'dropout_rate': 0., 'num_heads': 8}
#     transf_parameters_ohe = {'input_dim': 4, 'dropout_rate': 0.}
    
#     cls_module = CLSModel
#     att_module = TransformerNuc_onlyTransformer#TransformerNuc_onlyTransformer#TransformerNuc_onlyTransformer#TransformerNuc_onlyTransformer#TransformerNuc
#     ohe_module = Conv1DModel
    
#     labels_list = ['label']
#     equal_weights = torch.tensor([1.0, 1.0])
#     weight_dict = {label: equal_weights.to(device) for label in labels_list}

#     model_internal = Attention_Nucleosome_withCONV(cls_module, att_module, ohe_module ,device, transf_parameters_cls, transf_parameters_att,transf_parameters_ohe)

#     # Allenamento su train interno per selezione epoche
#     _, _, _, _, best_val_acc, best_model, best_epoch, _, _ = training_and_validation_loop_classification(
#         model_internal,
#         dataloader_internal_train,
#         dataloader_internal_val,
#         weight_dict,
#         epochs=300,
#         lr=1e-4,
#         patience=20
#     )

#     print(f"Best epoch for fold {fold+1}: {best_epoch}")

#     test_subset_cv = Subset(dataset_train, val_idx)
#     dataloader_test_cv = DataLoader(test_subset_cv, batch_size=64, shuffle=False)
    
#     train_subset_tot = Subset(dataset_train, train_idx)
#     dataloader_train_cv = DataLoader(train_subset_tot, batch_size=64, shuffle=True)
    
#     model_internal = Attention_Nucleosome_withCONV(cls_module, att_module, ohe_module ,device, transf_parameters_cls, transf_parameters_att,transf_parameters_ohe)

#     # Allenamento su train interno per selezione epoche
#     _, _, _, _, best_val_acc, best_model, best_epoch, _, _ = training_and_validation_loop_classification(
#         model_internal,
#         dataloader_train_cv,
#         dataloader_test_cv,
#         weight_dict,
#         epochs=int(best_epoch*1.),
#         lr=1e-4,#1e-6,
#         patience=int(best_epoch*1.)+1
#     )

#     fold_results.append({
#     'fold': fold + 1,
#     'test_metrics': test_classification(model_internal, dataloader_test_cv)[0],
#     'best_epoch': best_epoch
#     })

In [ ]:
0.7661
0.7597
0.743

In [ ]:
[i['test_metrics']['MCC'] for i in fold_results]

In [ ]:
for metrics in ['Sensitivity (Recall)',
  'Specificity',
  'Accuracy',
  'MCC',
  'AUC',
  'PR AUC',]:
    print(metrics, np.mean([i['test_metrics'][metrics] for i in fold_results]))



In [ ]:
epochs=300,
lr=1e-6,
patience=50

HS --> 0.76 con 2 teste dropout 0.0
HS --> 0.76 con 8 teste dropout 0.0
HS --> 0.75 con 2 teste e dropout 0.2

In [ ]:
assert False

In [ ]:
#CV FATTA BENE 

from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import Subset, DataLoader
import numpy as np
import torch

dim_embedding = 2560  # o 768 a seconda del tuo caso

dataset_train = Nuc_Dataset(dataset_train, dim_embedding)
labels = np.array([dataset_train[i]['label'].item() for i in range(len(dataset_train))])

skf = StratifiedKFold(n_splits=20, shuffle=True, random_state=42)

fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
    print(f"\n--- Fold {fold+1} ---")

    # Divisione fold in train/val
    train_labels = labels[train_idx]
    
    # Split interno train/test per selezione epoche
    internal_train_idx, internal_val_idx = train_test_split(
        train_idx,
        test_size=0.2,
        stratify=train_labels,
        random_state=fold  # diverso per ogni fold
    )

    internal_train_subset = Subset(dataset_train, internal_train_idx)
    internal_val_subset = Subset(dataset_train, internal_val_idx)
    
    dataloader_internal_train = DataLoader(internal_train_subset, batch_size=64, shuffle=True, collate_fn=collate_fn)
    dataloader_internal_val = DataLoader(internal_val_subset, batch_size=64, shuffle=False, collate_fn=collate_fn)

    # Modello + Pesi
    transf_parameters_cls = {'input_dim': dim_embedding, 'dropout_rate': 0.}
    transf_parameters_att = {'input_dim': dim_embedding, 'dropout_rate': 0.2, 'num_heads': 8}
    
    cls_module = CLSModel
    att_module = TransformerNuc#TransformerNuc_onlyTransformer#TransformerNuc
    labels_list = ['label']
    equal_weights = torch.tensor([1.0, 1.0])
    weight_dict = {label: equal_weights.to(device) for label in labels_list}

    model_internal = Attention_Nucleosome(cls_module, att_module, device, transf_parameters_cls, transf_parameters_att)

    # Allenamento su train interno per selezione epoche
    _, _, _, _, _, _, best_epoch, _, _ = training_and_validation_loop_classification(
        model_internal,
        dataloader_internal_train,
        dataloader_internal_val,
        weight_dict,
        epochs=300,
        lr=1e-5,
        patience=50
    )

    print(f"Best epoch for fold {fold+1}: {best_epoch}")

    # Riaddestramento su tutto il train (di fold) con best_epoch
    full_train_subset = Subset(dataset_train, train_idx)
    val_subset = Subset(dataset_train, val_idx)

    dataloader_full_train = DataLoader(full_train_subset, batch_size=64, shuffle=True, collate_fn=collate_fn)
    dataloader_val = DataLoader(val_subset, batch_size=64, shuffle=False, collate_fn=collate_fn)

    model_final = Attention_Nucleosome(cls_module, att_module, device, transf_parameters_cls, transf_parameters_att)

    # Addestramento finale con epoche scelte
    train_acc, val_acc, loss_train, loss_val, best_val_acc, best_model, _, final_output, final_labels = training_and_validation_loop_classification(
        model_final,
        dataloader_full_train,
        dataloader_val,
        weight_dict,
        epochs=round(best_epoch*1.2),
        lr=1e-5,
        patience=best_epoch  # o un numero più basso per early stopping disattivato
    )
    test_classification

    fold_results.append({
        'fold': fold + 1,
        'train_acc': train_acc,
        'val_acc': val_acc,
        'best_val_acc': best_val_acc,
        'best_epoch': best_epoch
    })

# Test finale (fuori dal loop)
dataloader_test = DataLoader(dataset_test, batch_size=64, shuffle=False, collate_fn=collate_fn)


In [ ]:
np.mean([i['val_acc'][-1] for i in fold_results])

In [ ]:
#hs77
#me67

In [ ]:
np.mean([i['best_val_acc'] for i in fold_results])

In [ ]:
test_classification

In [ ]:
assert False

In [ ]:
#TRAINO SU UNA SPECIE E TESTO SU UN' ALTRA

from sklearn.model_selection import StratifiedKFold
from torch.utils.data import Subset, DataLoader
import numpy as np
import torch

dim_embedding = 2560#768
fold_results=[]
dataset_train = Nuc_Dataset(dataset_train, dim_embedding)
dataset_test = Nuc_Dataset(dataset_test, dim_embedding)

dataloader_train = DataLoader(dataset_train, batch_size=64, shuffle=True, collate_fn=collate_fn)
dataloader_val = DataLoader(dataset_test, batch_size=64, shuffle=False, collate_fn=collate_fn)

# Pesi per la loss (restano uguali in ogni fold)
labels_list = ['label']
equal_weights = torch.tensor([1.0, 1.0])
weight_dict = {label: equal_weights.to(device) for label in labels_list}

# Parametri modello
transf_parameters_cls = {
    'input_dim': dim_embedding,
    'dropout_rate': 0.,
}

transf_parameters_att = {
    'input_dim': dim_embedding,
    'dropout_rate': 0.,
    'num_heads': 8
}

cls_module = CLSModel
att_module = TransformerNuc_onlyTransformer
Final_model = Attention_Nucleosome(cls_module, att_module, device, transf_parameters_cls, transf_parameters_att)

# Train + Val
train_acc, val_acc, loss_train, loss_val, best_val_acc, best_model, best_epoch, final_output, final_labels = training_and_validation_loop_classification(
    Final_model,
    dataloader_train,
    dataloader_val,
    weight_dict,
    epochs=90,
    lr=1e-6,
    patience=90
)

fold_results.append({
    'fold': fold + 1,
    'train_acc': train_acc,
    'val_acc': val_acc,
    'best_val_acc': best_val_acc,
    'best_epoch': best_epoch
})



In [ ]:
assert False

In [ ]:
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import Subset, DataLoader
import numpy as np
import torch

dim_embedding = 2560#768

dataset_train = Nuc_Dataset(dataset_train, dim_embedding)

# Etichette necessarie per StratifiedKFold
labels = np.array([dataset_train[i]['label'].item() for i in range(len(dataset_train))])

# 5-Fold Stratificato
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):

    print(f"\n--- Fold {fold+1} ---")

    train_subset = Subset(dataset_train, train_idx)
    val_subset = Subset(dataset_train, val_idx)

    dataloader_train = DataLoader(train_subset, batch_size=64, shuffle=True, collate_fn=collate_fn)
    dataloader_val = DataLoader(val_subset, batch_size=64, shuffle=False, collate_fn=collate_fn)

    # Pesi per la loss (restano uguali in ogni fold)
    labels_list = ['label']
    equal_weights = torch.tensor([1.0, 1.0])
    weight_dict = {label: equal_weights.to(device) for label in labels_list}

    # Parametri modello
    transf_parameters_cls = {
        'input_dim': dim_embedding,
        'dropout_rate': 0.2,
    }

    transf_parameters_att = {
        'input_dim': dim_embedding,
        'dropout_rate': 0.,
        'num_heads': 8
    }

    cls_module = CLSModel
    att_module = TransformerNuc
    Final_model = Attention_Nucleosome(cls_module, att_module, device, transf_parameters_cls, transf_parameters_att)

    # Train + Val
    train_acc, val_acc, loss_train, loss_val, best_val_acc, best_model, best_epoch, final_output, final_labels = training_and_validation_loop_classification(
        Final_model,
        dataloader_train,
        dataloader_val,
        weight_dict,
        epochs=210,
        lr=1e-5,
        patience=210
    )

    fold_results.append({
        'fold': fold + 1,
        'train_acc': train_acc,
        'val_acc': val_acc,
        'best_val_acc': best_val_acc,
        'best_epoch': best_epoch
    })

# Test finale
dataloader_test = DataLoader(dataset_test, batch_size=64, shuffle=False)


In [ ]:
np.mean([i['best_val_acc'] for i in fold_results])

In [ ]:
assert False

In [ ]:
#GPT CV


from sklearn.model_selection import KFold
from torch.utils.data import Subset, DataLoader
import numpy as np

dim_embedding = 768

dataset_train = Nuc_Dataset(dataset_train, dim_embedding)

dataset_test = Nuc_Dataset(dataset_test, dim_embedding)

# Etichette necessarie per stratificare se usi StratifiedKFold
labels = labels = np.array([dataset_train[i]['label'].item() for i in range(len(dataset_train))])
#np.array([dataset_train[i]['label'] for i in range(len(dataset_train))])

# 5-Fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

fold_results = []

for fold, (train_idx, val_idx) in enumerate(kf.split(dataset_train)):

    print(f"\n--- Fold {fold+1} ---")

    train_subset = Subset(dataset_train, train_idx)
    
    val_subset = Subset(dataset_train, val_idx)

    dataloader_train = DataLoader(train_subset, batch_size=8, shuffle=True, collate_fn=collate_fn)
    dataloader_val = DataLoader(val_subset, batch_size=8, shuffle=False, collate_fn=collate_fn)

    # Pesi per la loss (restano uguali in ogni fold)
    labels_list = ['label']
    equal_weights = torch.tensor([1.0, 1.0])
    weight_dict = {label: equal_weights.to(device) for label in labels_list}

    # Modello
    transf_parameters_cls = {
        'input_dim': 768,
        'dropout_rate': 0.1,
    }

    transf_parameters_att = {
        'input_dim': 768,
        'dropout_rate': 0.,
        'num_heads':8
    }


    #model = CLSModel#TransformerNuc

    cls_module= CLSModel
    att_module= TransformerNuc_onlyTransformer#TransformerNuc_onlyTransformer#TransformerNuc
    Final_model = Attention_Nucleosome(cls_module, att_module ,device,transf_parameters_cls, transf_parameters_att)#(model, device, **transf_parameters).to(device)

    # Train + Val
    train_acc, val_acc, loss_train, loss_val, best_val_acc, best_model, best_epoch, final_output, final_labels = training_and_validation_loop_classification(
        Final_model,
        dataloader_train,
        dataloader_val,
        weight_dict,
        epochs=300,
        lr=5e-6,
        patience=30
    )

    fold_results.append({
        'fold': fold + 1,
        'train_acc': train_acc,
        'val_acc': val_acc,
        'best_val_acc': best_val_acc,
        'best_epoch': best_epoch
    })

# Test finale su dataset_test (opzionale)
dataloader_test = DataLoader(dataset_test, batch_size=64, shuffle=False)


In [ ]:
from sklearn.model_selection import KFold
from torch.utils.data import Subset, DataLoader
import numpy as np
import torch

dim_embedding = 768

dataset_train = Nuc_Dataset(dataset_train, dim_embedding)
dataset_test = Nuc_Dataset(dataset_test, dim_embedding)

labels = np.array([dataset_train[i]['label'].item() for i in range(len(dataset_train))])

kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []

for fold, (train_val_idx, test_idx) in enumerate(kf.split(dataset_train)):

    print(f"\n--- Fold {fold+1} ---")

    # Nested KFold for internal validation
    inner_kf = KFold(n_splits=5, shuffle=True, random_state=fold)  # Different seed per outer fold
    inner_train_idx, inner_val_idx = next(inner_kf.split(train_val_idx))

    # Map inner indices to original indices
    mapped_inner_train_idx = [train_val_idx[i] for i in inner_train_idx]
    mapped_inner_val_idx = [train_val_idx[i] for i in inner_val_idx]

    # Subsets for inner training
    inner_train_subset = Subset(dataset_train, mapped_inner_train_idx)
    inner_val_subset = Subset(dataset_train, mapped_inner_val_idx)

    dataloader_inner_train = DataLoader(inner_train_subset, batch_size=64, shuffle=True)
    dataloader_inner_val = DataLoader(inner_val_subset, batch_size=64, shuffle=False)

    labels_list = ['label']
    equal_weights = torch.tensor([1.0, 1.0])
    weight_dict = {label: equal_weights.to(device) for label in labels_list}

    transf_parameters_cls = {
        'input_dim': 2560,
        'dropout_rate': 0.2,
    }

    transf_parameters_att = {
        'input_dim': 2560,
        'dropout_rate': 0.2,
        'num_heads': 8
    }

    cls_module = CLSModel
    att_module = TransformerNuc
    model = Attention_Nucleosome(cls_module, att_module, device, transf_parameters_cls, transf_parameters_att)

    # Inner training to find best epoch
    _, _, _, _, best_val_acc_inner, _, best_epoch, _, _ = training_and_validation_loop_classification(
        model,
        dataloader_inner_train,
        dataloader_inner_val,
        weight_dict,
        epochs=300,
        lr=1e-6,
        patience=30
    )

    print(f"  Best inner epoch: {best_epoch}")

    # Train full model on outer train_val_idx using best_epoch
    full_train_subset = Subset(dataset_train, train_val_idx)
    val_subset = Subset(dataset_train, test_idx)

    dataloader_train_full = DataLoader(full_train_subset, batch_size=64, shuffle=True)
    dataloader_val_outer = DataLoader(val_subset, batch_size=64, shuffle=False)

    final_model = Attention_Nucleosome(cls_module, att_module, device, transf_parameters_cls, transf_parameters_att)

    train_acc, val_acc, loss_train, loss_val, best_val_acc, best_model, _, final_output, final_labels = training_and_validation_loop_classification(
        final_model,
        dataloader_train_full,
        dataloader_val_outer,
        weight_dict,
        epochs=best_epoch,
        lr=1e-6,
        patience=best_epoch  # Early stopping disattivato
    )

    fold_results.append({
        'fold': fold + 1,
        'train_acc': train_acc,
        'val_acc': val_acc,
        'best_val_acc': best_val_acc,
        'best_epoch': best_epoch
    })

# Test finale (opzionale)
dataloader_test = DataLoader(dataset_test, batch_size=64, shuffle=False)


In [ ]:
np.mean([i['best_val_acc'] for i in fold_results])

In [ ]:
0.7654103344954631

In [ ]:
assert False

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

# Se sono tensori PyTorch, convertili prima
import torch

output = final_output['label']
labels = final_labels['label']

if isinstance(output, torch.Tensor):
    output = output.cpu().numpy().tolist()
if isinstance(labels, torch.Tensor):
    labels = labels.cpu().numpy().tolist()

# Matrice di confusione
cm = confusion_matrix(labels, output)
print("Confusion Matrix:")
print(cm)

# Visualizzazione con heatmap
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

# Report di classificazione
print("\nClassification Report:")
print(classification_report(labels, output, digits=4))

# Accuratezza
acc = accuracy_score(labels, output)
print(f"\nAccuracy: {acc:.4f}")


In [ ]:
len(final_labels['label'])